In [1]:
# Библиотеки и настройки
import pandas as pd
import numpy as np
import plotly.express as px
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Анализ таблицы продаж `sales`
Выполнила: Дюке Екатерина  
Дата: 28.06.2026

### Описание исследования
В рамках работы с клиентской базой аптечной сети был проведен RFM анализ, целью которого являлась сегментация клиентов и формирование базовых рекомендаций для отдела маркетинга по организации рассылок. Полученные результаты позволили выделить ключевые группы покупателей с разной степенью лояльности и активности, однако они не давали достаточной детализации для точечного подбора товарных предложений.  
Для повышения релевантности маркетинговых коммуникаций и формирования более персонализированных рекомендаций было принято решение дополнить RFM подход товарным анализом на основе детальных данных о продажах. Это позволит не только учитывать поведенческие характеристики клиентов, но и опираться на реальные паттерны спроса, чтобы предлагать конкретные товары, наиболее соответствующие потребностям каждой клиентской группы.

### Цели
Целью исследования является анализ продаж аптечной сети на основе выгрузки из таблицы `sales` для выявления ключевых закономерностей в ассортименте, ценах, выручке и структуре продаж по аптекам, дням недели и товарным позициям. Полученные результаты должны дополнить выводы RFM-анализа и стать основой для более точных рекомендаций по маркетинговым рассылкам, в том числе по подбору товарных предложений для разных клиентских сегментов.

### Задачи
1. Ознакомиться со структурой данных таблицы `sales`, определить состав полей, объем наблюдений, период покрытия и наличие пропусков.
2. Проверить качество данных и выявить возможные аномалии, ошибки и несоответствия в ключевых полях.
3. Подготовить данные к анализу и сформировать очищенную рабочую таблицу `sales_final`.
4. Изучить распределения основных показателей продаж и проверить базовые характеристики данных.
5. Оценить корректность связки между артикулом товара и его наименованием.
6. Проанализировать временную структуру продаж: пропущенные даты, дневные и часовые пики активности.
7. Исследовать ценовую структуру товаров, а также их маржинальность и вариативность цен.
8. Выполнить товарный анализ, чтобы определить принципы дальнейшего отбора товаров для маркетинговых рекомендаций.
9. Сформировать аналитическую основу для более точных и персонализированных рассылок после RFM-сегментации.

### Полученные результаты
[Перейти к краткому резюме](#scrollTo=mhnrK_rELr4t)  
[Перейти к рекомендациям для одноразовых покупателей](#scrollTo=pB_4jrDbfF5H)  
[Перейти к рекомендациям для остальных групп](#scrollTo=u40an8_W_xCb)

### Стек
Анализ выполняется в Python, поскольку выгрузка из базы была получена в формате CSV, а Python позволяет гибко и быстро работать с табличными данными, очищать их, проверять качество, строить агрегаты и визуализации, а также последовательно исследовать данные на всех этапах. Кроме того, при наличии ограниченного периода наблюдений - чуть больше месяца в `sales` - Python удобен для быстрой переборки гипотез, перепроверки расчетов и построения нескольких уровней аналитики без необходимости перегружать процесс отдельными ручными инструментами.   

### Имеющиеся данные
Таблица с информацией о продажах `sales`
- `DR_Dat` - дата покупки
- `DR_Tim` - время покупки
- `DR_NChk` - номер чека
- `DR_NDoc` - номер кассового документа
- `DR_apt` - номер магазина (FK - shops)
- `DR_Kkm` - номер кассового аппарата
- `DR_TDoc` - вид документа
- `DR_TPay` - форма платежа (18 - безнал, 15 - нал)
- `DR_CDrugs` - артикул товара
- `DR_NDrugs` - название товара
- `DR_Suppl` - поставщик
- `DR_Prod` - производитель
- `DR_Kol` - кол-во проданного товара
- `DR_CZak` - закупочная цена
- `DR_CRoz` - розничная цена
- `DR_SDisc` - сумма скидки
- `DR_CDisc` - код скидки
- `DR_BCDisc` - штрихкод скидки
- `DR_TabEmpl` - табельный номер сотрудника (FK - employee)
- `DR_Pos` - номер позиции в чеке
- `DR_VZak` - вид закупки (1 - обычный, 2 - интернет-заказ)

Несколько важных моментов:  
1. Одному чеку может соответствовать несколько строк в таблице продаж (1 товарная позиция = 1 строка). Чек можно определить как комбинацию `dr_apt`, `dr_nchk`, `dr_dat`.
2. Скидка (`DR_SDisc`) применяется не к каждой штуке товара (`DR_Kol`), а к итоговому количеству. Перемножать эти поля не нужно.

## Этап 1. Загрузка и знакомство с данными

In [2]:
sales = pd.read_csv("/content/drive/MyDrive/simulative/apteka/source_data/sales.csv")

In [3]:
# print("Информация о столбцах и их типах данных:")
# sales.info()

In [4]:
# Преобразование типов данных
sales['dr_dat'] = pd.to_datetime(sales['dr_dat'])
sales['dr_bcdisc'] = sales['dr_bcdisc'].astype(pd.Int64Dtype())
sales['dr_cdisc'] = sales['dr_cdisc'].astype(pd.Int64Dtype())

In [5]:
# Проверяем изменения
display(sales.info())
display(sales.head(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45128 entries, 0 to 45127
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   dr_dat      45128 non-null  datetime64[ns]
 1   dr_tim      45128 non-null  object        
 2   dr_nchk     45128 non-null  int64         
 3   dr_ndoc     45128 non-null  int64         
 4   dr_apt      45128 non-null  int64         
 5   dr_kkm      45128 non-null  int64         
 6   dr_tdoc     45128 non-null  object        
 7   dr_tpay     45128 non-null  int64         
 8   dr_cdrugs   45128 non-null  int64         
 9   dr_ndrugs   45128 non-null  object        
 10  dr_suppl    45128 non-null  object        
 11  dr_prod     45128 non-null  object        
 12  dr_kol      45128 non-null  float64       
 13  dr_czak     45128 non-null  float64       
 14  dr_croz     45128 non-null  float64       
 15  dr_sdisc    45128 non-null  float64       
 16  dr_cdisc    17936 non-

None

,dr_dat,dr_tim,dr_nchk,dr_ndoc,dr_apt,dr_kkm,dr_tdoc,dr_tpay,dr_cdrugs,dr_ndrugs,dr_suppl,dr_prod,dr_kol,dr_czak,dr_croz,dr_sdisc,dr_cdisc,dr_bcdisc,dr_tabempl,dr_pos,dr_vzak
0,2022-05-01,08:33:09,6251,7001554,7,22553,Розничная реализация,18,20398,"ОТРИВИН 0,1% 10МЛ. №1 НАЗАЛ.СПРЕЙ ФЛ. /ГЛАКСО/НОВАРТИС/",Фармкомплект ООО,Новартис Консьюмер Хелс С.А.,1.0,146.0,175.0,8.0,9,200010024843,37,1,1


Пропуски только в типе скидки и штрихкоде, нужно проверить, является ли это признаком того, что просто не применялась скидка.

In [6]:
print(f"Период данных: от {sales['dr_dat'].min().date()} до {sales['dr_dat'].max().date()}")

# Количество пропущенных значений в каждом столбце
missing_cdisc = sales['dr_cdisc'].isnull().sum()
missing_bcdisc = sales['dr_bcdisc'].isnull().sum()

print(f"Пропущено значений в dr_cdisc: {missing_cdisc}")
print(f"Пропущено значений в dr_bcdisc: {missing_bcdisc}")

# Количество строк, где пропущены значения в обоих столбцах
both_missing = sales['dr_cdisc'].isnull() & sales['dr_bcdisc'].isnull()
count_both_missing = both_missing.sum()
print(f"Пропущено значений в обоих столбцах (dr_cdisc и dr_bcdisc): {count_both_missing}")

# Количество строк, где 'dr_cdisc' пропущено, а 'dr_bcdisc' заполнено
cdisc_missing_bcdisc_present = sales['dr_cdisc'].isnull() & sales['dr_bcdisc'].notnull()
count_cdisc_missing_bcdisc_present = cdisc_missing_bcdisc_present.sum()
print(f"dr_cdisc пропущено, dr_bcdisc заполнено: {count_cdisc_missing_bcdisc_present}")

# Количество строк, где 'dr_bcdisc' пропущено, а 'dr_cdisc' заполнено
bcdisc_missing_cdisc_present = sales['dr_bcdisc'].isnull() & sales['dr_cdisc'].notnull()
count_bcdisc_missing_cdisc_present = bcdisc_missing_cdisc_present.sum()
print(f"dr_bcdisc пропущено, dr_cdisc заполнено: {count_bcdisc_missing_cdisc_present}")

# dr_sdisc > 0, но dr_cdisc пропущен
anomalous_sdisc = sales[(sales['dr_sdisc'] > 0) & sales['dr_cdisc'].isnull()]
print(f"Проверка, что dr_sdisc > 0, но dr_cdisc пропущен: {len(anomalous_sdisc)} строк ({len(anomalous_sdisc)/len(sales)*100:.2f}%)")
if len(anomalous_sdisc) > 0:
    print(f"Примеры:")
    print(anomalous_sdisc[['dr_sdisc', 'dr_ndrugs', 'dr_kol', 'dr_cdisc', 'dr_bcdisc', 'dr_apt', 'dr_dat']].head(5))
else:
    print(f"КОРРЕКТНО, если в типе скидки пропуск, то ее сумма равна 0")

Период данных: от 2022-05-01 до 2022-06-09
Пропущено значений в dr_cdisc: 27192
Пропущено значений в dr_bcdisc: 27192
Пропущено значений в обоих столбцах (dr_cdisc и dr_bcdisc): 27192
dr_cdisc пропущено, dr_bcdisc заполнено: 0
dr_bcdisc пропущено, dr_cdisc заполнено: 0
Проверка, что dr_sdisc > 0, но dr_cdisc пропущен: 0 строк (0.00%)
КОРРЕКТНО, если в типе скидки пропуск, то ее сумма равна 0


Добавим столбец с идентификатором чека (комбинация `dr_apt`, `dr_nchk`, `dr_dat`)

In [7]:
sales['check_id'] = sales['dr_apt'].astype(str) + '_' + sales['dr_nchk'].astype(str) + '_' + sales['dr_dat'].dt.strftime('%Y%m%d')

Добавим столбец с названиями аптек `shop`  

С помощью словаря `shop_mapping` (выгружен из базы, таблица `shops`) добавлен столбец `shop`: по `dr_apt` (ID аптеки) подставлено понятное название. Это повышает читаемость отчетов и упрощает сегментацию по точкам продаж.  

*Справочник аптек (`id` -> `shop`)*
| id | name |
| --- | --- |
| 2 | Аптека 2 |
| 6 | Аптека 3 |
| 7 | Аптека 4 |
| 9 | Аптека 5 |
| 10 | Аптека 6 |
| 11 | Аптека 7 |
| 13 | Аптека 1 |
| 15 | Аптека 8 |
| 17 | Аптека 10 |
| 18 | Аптека 11 |  

In [8]:
shop_mapping = {
    2: 'Аптека 2',
    6: 'Аптека 3',
    7: 'Аптека 4',
    9: 'Аптека 5',
    10: 'Аптека 6',
    11: 'Аптека 7',
    13: 'Аптека 1',
    15: 'Аптека 8',
    17: 'Аптека 10',
    18: 'Аптека 11'
}

sales['shop'] = sales['dr_apt'].map(shop_mapping)
print(f"Список аптек: {sales['shop'].unique()}")
display(sales[['dr_apt', 'shop']].head())

Список аптек: ['Аптека 4' 'Аптека 3' 'Аптека 2' 'Аптека 8' 'Аптека 7' 'Аптека 11'
 'Аптека 10' 'Аптека 1']


,dr_apt,shop
0,7,Аптека 4
1,6,Аптека 3
2,2,Аптека 2
3,2,Аптека 2
4,7,Аптека 4


- Аптеки 6 нет, она похоже закрылась после октября 2021
- Аптека 3 отсутсвует в `bonuscheques`
- Совпадают с `bonuscheques` 1, 2, 4, 7, 8, 10, 11. Аптека 11, наоборот, только открылась в марте 2022

In [9]:
# Проверим изменения
print(sales.columns)
print(f"Пример идентификатора чека: {sales['check_id'].head(1)}")

Index(['dr_dat', 'dr_tim', 'dr_nchk', 'dr_ndoc', 'dr_apt', 'dr_kkm', 'dr_tdoc',
       'dr_tpay', 'dr_cdrugs', 'dr_ndrugs', 'dr_suppl', 'dr_prod', 'dr_kol',
       'dr_czak', 'dr_croz', 'dr_sdisc', 'dr_cdisc', 'dr_bcdisc', 'dr_tabempl',
       'dr_pos', 'dr_vzak', 'check_id', 'shop'],
      dtype='object')
Пример идентификатора чека: 0    7_6251_20220501
Name: check_id, dtype: object


### 🕵🏻 Наблюдение 1
- В датасете 45 128 записей и 21 столбец.
- Данные покрывают период с 1 мая по 9 июня 2022 года. Период в `bonuscheques` с 12 июля 2021 года по 9 июня 2022 года.
- В столбцах `dr_cdisc` и `dr_bcdisc` пропущено по 27 192 значения (около 60%). При этом пропуски полностью совпадают: нет случаев, когда один из столбцов заполнен, а другой нет. Это говорит о системной логике заполнения: скидка либо есть (и заполнены оба поля), либо нет (оба поля пусты), а в сумма скидки 0.
- Дата покупки (`dr_dat`) приведена к типу datetime - это позволяет корректно работать с временными интервалами. Поля `dr_cdisc` (код скидки) и `dr_bcdisc` (штрихкод скидки) переведены в целочисленный тип с поддержкой пропусков - это упрощает дальнейшую агрегацию и избегает ошибок при сравнении.
- Создан столбец `check_id` на основе комбинации `dr_apt` (номер магазина), `dr_nchk` (номер чека) и `dr_dat` (дата покупки). Это позволяет однозначно группировать строки по чекам и корректно агрегировать данные (например, считать общую сумму чека или анализировать товарные наборы). Пример идентификатора: `7_6251_20220501`.  
- Набор аптек в текущих продажах не полностью совпадает с данными из таблицы `bonuscheques`. Совпадают: Аптеки 1,2,4,7,8,10,11 / Аптека 3 отсутствует в `bonuscheques`.
  - Аптека 6 скорее всего закрыта еще до периода анализа (после октября 2021) - ее нет в текущих продажах, что логично.
  - Аптека 11 скорее всего открылась в марте 2022 - она присутствует в данных `sales`, что подтверждает их актуальность и корректность временного среза.
  - Аптека 3 есть в `sales`, но нет в `bonuscheques` — возможны разные причины (от отсутствия бонусной программы в точке до расхождений в id и правилах выгрузки), но по текущим данным нельзя точно определить причину.

Данные приведены к аналитически пригодному виду: исправлены типы, проверена целостность полей скидок, обеспечена возможность работы на уровне чеков. Это создает надежную основу для товарного анализа и сопоставления с результатами RFM сегментации.

### Проверка качества данных и аномалий

In [10]:
sales.head(1)

,dr_dat,dr_tim,dr_nchk,dr_ndoc,dr_apt,dr_kkm,dr_tdoc,dr_tpay,dr_cdrugs,dr_ndrugs,dr_suppl,dr_prod,dr_kol,dr_czak,dr_croz,dr_sdisc,dr_cdisc,dr_bcdisc,dr_tabempl,dr_pos,dr_vzak,check_id,shop
0,2022-05-01,08:33:09,6251,7001554,7,22553,Розничная реализация,18,20398,"ОТРИВИН 0,1% 10МЛ. №1 НАЗАЛ.СПРЕЙ ФЛ. /ГЛАКСО/НОВАРТИС/",Фармкомплект ООО,Новартис Консьюмер Хелс С.А.,1.0,146.0,175.0,8.0,9,200010024843,37,1,1,7_6251_20220501,Аптека 4


In [11]:
# Отрицательные значения в dr_kol,	dr_czak,	dr_croz
neg_kol = sales[sales['dr_kol'] < 0]
print(f"Отрицательные dr_kol: {len(neg_kol)} строк ({len(neg_kol)/len(sales)*100:.2f}% от всех)")

neg_czak = sales[sales['dr_czak'] < 0]
print(f"Отрицательные dr_czak: {len(neg_czak)} строк ({len(neg_czak)/len(sales)*100:.2f}%)")

neg_croz = sales[sales['dr_croz'] < 0]
print(f"Отрицательные dr_croz: {len(neg_croz)} строк ({len(neg_croz)/len(sales)*100:.2f}%)")

Отрицательные dr_kol: 4 строк (0.01% от всех)
Отрицательные dr_czak: 0 строк (0.00%)
Отрицательные dr_croz: 0 строк (0.00%)


In [12]:
# Посмотрим dr_kol
if len(neg_kol) > 0:
    print(f"Минимальное dr_kol: {neg_kol['dr_kol'].min()}")
    print(f"Примеры:")
    display(neg_kol[['dr_dat', 'dr_tim', 'dr_kol', 'dr_ndrugs', 'dr_croz', 'dr_czak', 'dr_sdisc', 'dr_tdoc', 'dr_apt']])

Минимальное dr_kol: -1.0
Примеры:


,dr_dat,dr_tim,dr_kol,dr_ndrugs,dr_croz,dr_czak,dr_sdisc,dr_tdoc,dr_apt
17326,2022-05-17,12:50:33,-1.0,ДИНАМИКА УСТРОЙСТВО Д/ИНЪЕКЦИЙ,853.0,613.29,-102.0,Розничная реализация,6
19118,2022-05-18,17:47:25,-1.0,КОМПИД НАБОР ПЛАСТЫРЕЙ П/ВЛАЖНЫХ МОЗОЛЕЙ №5 (МАЛ.№2+СРЕД.№2+БОЛ.№1) [COMPEED],722.0,534.35,0.0,Розничная реализация,18
21263,2022-05-20,14:27:53,-1.0,СОЛГАР КОЭНЗИМ Q-10 60МГ. №30 КАПС. [SOLGAR],1848.0,1420.85,0.0,Розничная реализация,18
31844,2022-05-29,14:08:58,-1.0,ОМРОН ТОНОМЕТР M2 КЛАССИК УНИВЕРС.МАНЖЕТА /АРТ.HEM-7122-LRU/ [OMRON],4702.0,3432.00,-470.0,Розничная реализация,13


In [13]:
if len(neg_kol) > 0:
    # Извлекаем уникальные комбинации признаков из возвратов
    return_features = neg_kol[['dr_ndrugs', 'dr_croz', 'dr_czak', 'dr_tdoc', 'dr_apt']].drop_duplicates()

    # Ищем соответствующие покупки в sales, где dr_kol > 0
    matching_purchases = sales.merge(return_features, on=['dr_ndrugs', 'dr_croz', 'dr_czak', 'dr_tdoc', 'dr_apt'], how='inner')

    # Дополнительно фильтруем, чтобы убедиться, что dr_kol положительное (хотя sales_no_returns уже это делает)
    matching_purchases = matching_purchases[matching_purchases['dr_kol'] > 0]

    print(f"\nНайдено {len(matching_purchases)} строк, которые могут быть оригинальными покупками для возвратов:")
    display(matching_purchases[['dr_dat', 'dr_tim', 'dr_kol', 'dr_ndrugs', 'dr_croz', 'dr_czak', 'dr_sdisc', 'dr_tdoc', 'dr_apt']])
else:
    print("Нет отрицательных значений dr_kol для поиска соответствующих покупок.")


Найдено 3 строк, которые могут быть оригинальными покупками для возвратов:


,dr_dat,dr_tim,dr_kol,dr_ndrugs,dr_croz,dr_czak,dr_sdisc,dr_tdoc,dr_apt
0,2022-05-17,12:29:58,1.0,ДИНАМИКА УСТРОЙСТВО Д/ИНЪЕКЦИЙ,853.0,613.29,102.0,Розничная реализация,6
3,2022-05-20,12:54:12,1.0,ДИНАМИКА УСТРОЙСТВО Д/ИНЪЕКЦИЙ,853.0,613.29,102.0,Розничная реализация,6
4,2022-05-20,13:16:44,1.0,СОЛГАР КОЭНЗИМ Q-10 60МГ. №30 КАПС. [SOLGAR],1848.0,1420.85,0.0,Розничная реализация,18


In [14]:
print("Поиск оригинальных покупок для конкретных возвратов:")

# Определяем условия для первого товара с использованием np.isclose для float значений
condition_item1 = (
    (sales['dr_ndrugs'] == 'КОМПИД НАБОР ПЛАСТЫРЕЙ П/ВЛАЖНЫХ МОЗОЛЕЙ №5 (МАЛ.№2+СРЕД.№2+БОЛ.№1) [COMPEED]') &
    (sales['dr_tdoc'] == 'Розничная реализация') &
    (sales['dr_apt'] == 18) &
    (sales['dr_kol'] > 0)
)

# Определяем условия для второго товара с использованием np.isclose для float значений
condition_item2 = (
    (sales['dr_ndrugs'] == 'ОМРОН ТОНОМЕТР M2 КЛАССИК УНИВЕРС.МАНЖЕТА /АРТ.HEM-7122-LRU/ [OMRON]') &
    (sales['dr_tdoc'] == 'Розничная реализация') &
    (sales['dr_apt'] == 13) &
    (sales['dr_kol'] > 0)
)

# Объединяем условия и применяем к датафрейму sales
specific_purchases = sales[condition_item1 | condition_item2]

display(specific_purchases[['dr_dat', 'dr_tim', 'dr_kol', 'dr_ndrugs', 'dr_croz', 'dr_czak', 'dr_sdisc', 'dr_tdoc', 'dr_apt']])

Поиск оригинальных покупок для конкретных возвратов:


,dr_dat,dr_tim,dr_kol,dr_ndrugs,dr_croz,dr_czak,dr_sdisc,dr_tdoc,dr_apt


Два - возвраты, еще два не смогла точно подтвердить, возможно и техническая ошибка  
❌ **Не включать в аналитику продаж**

In [15]:
# Отрицательные скидки dr_sdisc
neg_sdisc = sales[sales['dr_sdisc'] < 0]
print(f"Отрицательные dr_sdisc: {len(neg_sdisc)} строк ({len(neg_sdisc)/len(sales)*100:.2f}%)")

Отрицательные dr_sdisc: 3 строк (0.01%)


In [16]:
# Посмотрим dr_sdisc
if len(neg_sdisc) > 0:
    print(f"Минимальная dr_sdisc: {neg_sdisc['dr_sdisc'].min()}")
    print(f"Примеры:")
    display(neg_sdisc[['dr_sdisc', 'dr_ndrugs', 'dr_kol', 'dr_cdisc', 'dr_bcdisc', 'dr_apt']])

Минимальная dr_sdisc: -470.0
Примеры:


,dr_sdisc,dr_ndrugs,dr_kol,dr_cdisc,dr_bcdisc,dr_apt
2898,-0.01,ПАКЕТ,1.0,<NA>,<NA>,2
17326,-102.00,ДИНАМИКА УСТРОЙСТВО Д/ИНЪЕКЦИЙ,-1.0,<NA>,<NA>,6
31844,-470.00,ОМРОН ТОНОМЕТР M2 КЛАССИК УНИВЕРС.МАНЖЕТА /АРТ.HEM-7122-LRU/ [OMRON],-1.0,<NA>,<NA>,13


Выделяется только пакет, похоже на техническую ошибку, остальное возвраты  
❌ **Не включать в аналитику продаж**

In [17]:
# Несоответствие закупочной и розничной цены dr_czak > dr_croz
price_anomaly = sales[sales['dr_czak'] > sales['dr_croz']]
print(f"Случаи dr_czak > dr_croz: {len(price_anomaly)} строк ({len(price_anomaly)/len(sales)*100:.2f}%)")
if len(price_anomaly) > 0:
    print(f"Максимальная разница (zak - roz): {(price_anomaly['dr_czak'] - price_anomaly['dr_croz']).max()}")
    print(f"Примеры:")
    display(price_anomaly[['dr_czak', 'dr_croz', 'dr_ndrugs', 'dr_cdrugs', 'dr_kol', 'dr_sdisc', 'dr_apt', 'dr_dat']].head())

Случаи dr_czak > dr_croz: 52 строк (0.12%)
Максимальная разница (zak - roz): 34.67
Примеры:


,dr_czak,dr_croz,dr_ndrugs,dr_cdrugs,dr_kol,dr_sdisc,dr_apt,dr_dat
205,59.67,25.0,Карта LOYALITY 25Р,1504330,1.0,0.0,15,2022-05-01
1866,59.67,25.0,Карта LOYALITY 25Р,1504330,1.0,0.0,15,2022-05-02
4958,59.67,25.0,Карта LOYALITY 25Р,1504330,1.0,0.0,18,2022-05-05
5000,59.67,25.0,Карта LOYALITY 25Р,1504330,1.0,0.0,2,2022-05-05
7353,59.67,25.0,Карта LOYALITY 25Р,1504330,1.0,0.0,2,2022-05-07


Аптеки продают карты лоялности и их цена выше закупочной, что выглядит естественно. Нужно будет проверить, есть ли еще варианты карт лояльности (просто закупочная ниже розничной)  
❌ **Не включать продажи карт лояльности в аналитику продаж, но сохранить их в отдельный датафрейм, анализ программы лояльности**

In [18]:
# Также проверим равенство цен (маржа = 0)
zero_margin = sales[sales['dr_czak'] == sales['dr_croz']]
print(f"Случаи dr_czak == dr_croz (маржа = 0): {len(zero_margin)} строк ({len(zero_margin)/len(sales)*100:.2f}%)")

# Товары с маржой = 0
zero_margin_items = sales[sales['dr_czak'] == sales['dr_croz']]
display(zero_margin_items['dr_ndrugs'].value_counts())

Случаи dr_czak == dr_croz (маржа = 0): 269 строк (0.60%)


,count
dr_ndrugs,
"Карта LOYALITY 0,01Р",268
МУКАЛТИН РЕНЕВАЛ 50МГ. №20 ТАБ. /ОБНОВЛЕНИЕ/,1


- В основном карты лояльности, котрые нужно исключить
- МУКАЛТИН РЕНЕВАЛ 50МГ. №20 ТАБ. /ОБНОВЛЕНИЕ/ был единожды продан по закупочной цене, что не повлияет на товарный анализ

#### Удалим строки. Датафрейм для товарного анализа `sales_final`
Исключаем:  
- возвраты (Отрицательные `dr_kol`: 4 строки)
- отрицательные `dr_sdisc` + техническую ошибку (3 строки, включая ПАКЕТ)

In [19]:
# Фиксируем исходный размер
original_size = len(sales)
print(f"Исходное количество строк: {original_size}")

# Исключаем возвраты (Отрицательные dr_kol: 4 строки)
returns = sales[sales['dr_kol'] < 0]
sales_no_returns = sales[sales['dr_kol'] >= 0]
print(f"Возвратов исключено: {len(returns)} строк ({len(returns)/original_size*100:.2f}%)")

# Исключаем техническую ошибку и отрицательные скидки
negative_sdisc = sales_no_returns[sales_no_returns['dr_sdisc'] < 0]
sales_cleaned = sales_no_returns[sales_no_returns['dr_sdisc'] >= 0]
print(f"Технических ошибок исключено: {len(negative_sdisc)} строк ({len(negative_sdisc)/original_size*100:.5f}%)")

# Исключить карты лояльности
cards = sales[sales['dr_ndrugs'].str.contains('LOYALITY', case=False, na=False)]
sales_no_cards = sales_cleaned[sales_cleaned['dr_ndrugs'].str.contains('LOYALITY', case=False, na=False) == False]
print(f"Карт лояльности исключено: {len(cards)} строк ({len(cards)/original_size*100:.2f}%)")

# Финальный очищенный датафрейм для товарного анализа
sales_final = sales_no_cards

# Зафиксировать % убранных данных
total_removed = original_size - len(sales_final)
percentage_removed = total_removed / original_size * 100

Исходное количество строк: 45128
Возвратов исключено: 4 строк (0.01%)
Технических ошибок исключено: 1 строк (0.00222%)
Карт лояльности исключено: 320 строк (0.71%)


#### Датафрейм для товарного анализа `sales_final`

In [20]:
print("=== ФИНАЛЬНАЯ ОЧИСТКА: Сводка ===")
print(f"\nИсходное количество строк: {original_size}")
print(f"Убрано строк: {total_removed}")
print(f"Убранных данных: {percentage_removed:.2f}%")
print(f"\nОчищенное количество строк для товарного анализа: {len(sales_final)}")
print(f"Оставшихся данных: {100 - percentage_removed:.2f}%")

=== ФИНАЛЬНАЯ ОЧИСТКА: Сводка ===

Исходное количество строк: 45128
Убрано строк: 325
Убранных данных: 0.72%

Очищенное количество строк для товарного анализа: 44803
Оставшихся данных: 99.28%


### 🕵🏻 Наблюдение 2
- Найдено 4 строки с отрицательным `dr_kol` (0.01%), в основном подтверждены как возвраты (есть соответствующие покупки с положительным количеством). Логично исключить из анализа продаж.
- Найдено 3 строки с отрицательной скидкой (0.01%), включая техническую ошибку по товару ПАКЕТ, логично исключить из анализа продаж.
- 320 строк (0.71%) с аномальной маржой (в т.ч. `dr_czak > dr_croz` и `dr_czak == dr_croz`) - это специфика продукта (продажа карт лояльности), а не ошибка. Для товарного анализа их нужно убрать, но можно выделить в отдельный датасет для оценки программы лояльности.
- Проверка на нулевую маржу дала 269 строк, преимущественно карты лояльности. Единственная позиция вне этой категории (Мукалтин) - единичный случай, не искажает общую картину.  

**Финальный датафрейм для товарного анализа: `sales_final` (44 803 строки, 99.28% от исходных данных)**.

#### Аптеки продают карты лояльности
Соберем их в отдельный датафрейм, чтобы посмотреть по аптекам, где хорошо их продают, а где хуже (поможет в оценке программы лояльности аптечной сети)

In [21]:
# Все товары с "LOYALITY" в названии (без учета регистра)
sales['has_karta'] = sales['dr_ndrugs'].str.contains('LOYALITY', case=False, na=False)
cards_df = sales[sales['has_karta']]
print(f"Всего продаж карт лояльности найдено: {len(cards_df)} строк ({len(cards_df)/len(sales)*100:.2f}% от всех)")

# Количество записей по каждому названию карты
print("\n Продажи карт по названию (количество записей):")
cards_by_name = cards_df['dr_ndrugs'].value_counts().reset_index()
cards_by_name.columns = ['card_name', 'count']
display(cards_by_name)

Всего продаж карт лояльности найдено: 320 строк (0.71% от всех)

 Продажи карт по названию (количество записей):


,card_name,count
0,"Карта LOYALITY 0,01Р",268
1,Карта LOYALITY 25Р,52


In [22]:
# Суммарное количество карт по аптекам (с учетом dr_kol)
print("Суммарное количество карт по аптекам (dr_kol):")
cards_by_shop_kol = cards_df.groupby('shop')['dr_kol'].sum().reset_index()
cards_by_shop_kol.columns = ['shop', 'cards_total_kol']
cards_by_shop_kol = cards_by_shop_kol.sort_values('cards_total_kol', ascending=False)
display(cards_by_shop_kol)
print(f"Сумма по dr_kol: {cards_by_shop_kol['cards_total_kol'].sum()}")

# Дополнительная проверка: Найдем записи о картах, где dr_kol не равно 1
anomalous_card_kol = cards_df[cards_df['dr_kol'] != 1]
if not anomalous_card_kol.empty:
    print("\nОбнаружены записи о продаже карт лояльности, где dr_kol не равно 1:")
    display(anomalous_card_kol[['dr_dat', 'dr_tim', 'dr_ndrugs', 'dr_kol', 'dr_croz', 'dr_sdisc', 'shop']])
else:
    print("\nВсе записи о продаже карт лояльности имеют dr_kol = 1.")

Суммарное количество карт по аптекам (dr_kol):


,shop,cards_total_kol
2,Аптека 11,114.0
3,Аптека 2,72.0
1,Аптека 10,47.0
6,Аптека 8,36.0
5,Аптека 7,23.0
0,Аптека 1,18.0
4,Аптека 4,9.0


Сумма по dr_kol: 319.0

Обнаружены записи о продаже карт лояльности, где dr_kol не равно 1:


,dr_dat,dr_tim,dr_ndrugs,dr_kol,dr_croz,dr_sdisc,shop
43988,2022-06-09,10:47:50,"Карта LOYALITY 0,01Р",0.290274,0.01,0.0,Аптека 10
43989,2022-06-09,10:47:50,"Карта LOYALITY 0,01Р",0.709726,0.01,0.0,Аптека 10


Похоже на техническую ошибку. Одна продажа карты лояльности была разделена на две строки с дробными количествами, что и привело к тому, что у нас 320 строк, но общее количество `dr_kol` составляет 319.  
**Считать продажи карт лояльности по количеству строк**

In [23]:
specific_loyalty_cards = [
    'Карта LOYALITY 0,01Р',
    'Карта LOYALITY 25Р'
]

# Фильтруем cards_df по указанным названиям карт
filtered_cards = cards_df[cards_df['dr_ndrugs'].isin(specific_loyalty_cards)]

# Группируем по аптеке и названию карты, считаем количество строк
distribution_by_shop_and_card = filtered_cards.groupby(['shop', 'dr_ndrugs']).size().unstack(fill_value=0)

print("Распределение карт лояльности по аптекам (количество строк):")
display(distribution_by_shop_and_card)

Распределение карт лояльности по аптекам (количество строк):


dr_ndrugs,"Карта LOYALITY 0,01Р",Карта LOYALITY 25Р
shop,,
Аптека 1,18,0
Аптека 10,46,2
Аптека 11,88,26
Аптека 2,64,8
Аптека 4,6,3
Аптека 7,19,4
Аптека 8,27,9


### 🕵🏻 Наблюдение 3
- Все продажи карт лояльности выделены в отдельный датафрейм `cards_df` по наличию подстроки `«LOYALITY»` в названии товара (`dr_ndrugs`).
- Найдена аномалия - одну продажу разбили на две строки с дробными значениями, из‑за чего общее количество карт по `dr_kol` (319) не совпадает с числом строк (320). Для оценки активности аптек решили использовать подсчет по количеству строк.
- Найдено 2 типа карт: Карта LOYALITY 0,01Р (268 строк) и Карта LOYALITY 25Р (5 строки).
- Аптеки 3 нет в продажах карт лояльности - согласуется с тем, что ее нет и в таблице `bonuscheques`. Это усиливает гипотезу о том, что точка либо не участвовала в программе лояльности, либо данные по ней не попадали в витрины программы.
- Аптека 11 лидирует по продажам обоих типов карт и в целом. Она выдеяется среди аптек именно наибольшим числом проданных Карта LOYALITY 25Р (vip-карты с большой скидкой, которую продают за деньги, а не дают бесплатно).
- Аптека 2 также активно продает карты, особенно Карта LOYALITY 0,01Р.
- Аптека 10 так же имеет значительное количество проданных карт обоих видов.
- Хуже всего карты продаются в Аптеке 4.

### Проверка связки артикул–название


In [24]:
df = sales_final.copy()

# 1 артикул - 1 название и наоборот
sku_names = df.groupby('dr_cdrugs')['dr_ndrugs'].nunique().sort_values(ascending=False)
name_skus = df.groupby('dr_ndrugs')['dr_cdrugs'].nunique().sort_values(ascending=False)

sku_more_1_name = sku_names[sku_names > 1]
name_more_1_sku = name_skus[name_skus > 1]

# Уникальные артикулы и названия
n_unique_sku = df['dr_cdrugs'].nunique()
n_unique_name = df['dr_ndrugs'].nunique()

print("=== Артикулы / названия ===")
print(f"Уникальных артикулов: {n_unique_sku}")
print(f"Уникальных названий: {n_unique_name}")
print(f"Артикулов с >1 названием: {len(sku_more_1_name)}")
print(f"Названий с >1 артикулом: {len(name_more_1_sku)}")

print("Примеры артикулов с >1 названием:")
display(sku_more_1_name.head(5).reset_index(name='n_names'))

print("Примеры названий с >1 артикулом:")
display(name_more_1_sku.reset_index(name='n_skus'))

=== Артикулы / названия ===
Уникальных артикулов: 6499
Уникальных названий: 6557
Артикулов с >1 названием: 60
Названий с >1 артикулом: 2
Примеры артикулов с >1 названием:


,dr_cdrugs,n_names
0,336283,2
1,816,2
2,304140,2
3,12555,2
4,148623,2


Примеры названий с >1 артикулом:


,dr_ndrugs,n_skus
0,ПЕРЕКИСЬ ВОДОРОДА 3% 100МЛ. Р-Р ФЛ. ПЛАСТ. /ТУЛЬСКАЯ ФФ/,2
1,ДОКТОР МОМ ФИТО 20Г. МАЗЬ Д/НАРУЖ.ПРИМ. БАНКА,2


В данных есть неоднозначности между артикулом и названием, проверим

In [25]:
# Проблемные названия
problem_names = [
    "ДОКТОР МОМ ФИТО 20Г. МАЗЬ Д/НАРУЖ.ПРИМ. БАНКА",
    "ПЕРЕКИСЬ ВОДОРОДА 3% 100МЛ. Р-Р ФЛ. ПЛАСТ. /ТУЛЬСКАЯ ФФ/"
]

for name in problem_names:
    print(f"\n{'='*80}")
    print(f"ТОВАР: {name}")
    print(f"{'='*80}")

    tmp = sales_final[sales_final['dr_ndrugs'] == name].copy()

    # Сводка по артикулам
    summary = (
        tmp.groupby('dr_cdrugs')
        .agg(
            n_rows=('dr_cdrugs', 'size'),
            first_date=('dr_dat', 'min'),
            last_date=('dr_dat', 'max'),
            shops=('shop', lambda x: ', '.join(sorted(x.astype(str).unique()))),
            suppliers=('dr_suppl', lambda x: ', '.join(sorted(x.dropna().astype(str).unique()))),
            producers=('dr_prod', lambda x: ', '.join(sorted(x.dropna().astype(str).unique()))),
            zak_min=('dr_czak', 'min'),
            zak_max=('dr_czak', 'max'),
            roz_min=('dr_croz', 'min'),
            roz_max=('dr_croz', 'max')
        )
        .reset_index()
        .sort_values(['first_date', 'dr_cdrugs'])
    )

    display(summary)

    # Динамика по датам и аптекам
    by_date_shop = (
        tmp.groupby(['dr_dat', 'shop', 'dr_cdrugs'])
        .agg(
            n_rows=('dr_cdrugs', 'size'),
            qty=('dr_kol', 'sum'),
            zak=('dr_czak', 'mean'),
            roz=('dr_croz', 'mean')
        )
        .reset_index()
        .sort_values(['dr_dat', 'shop', 'dr_cdrugs'])
    )

    print("Строки по датам и аптекам:")
    display(by_date_shop)


ТОВАР: ДОКТОР МОМ ФИТО 20Г. МАЗЬ Д/НАРУЖ.ПРИМ. БАНКА


,dr_cdrugs,n_rows,first_date,last_date,shops,suppliers,producers,zak_min,zak_max,roz_min,roz_max
1,336283,1,2022-05-05,2022-05-05,Аптека 11,Протек,Юник Фармасьютикал Лабораториз/Дж Б Кемикалс Фарма,182.18,182.18,250.0,250.0
0,28901,10,2022-05-19,2022-06-08,"Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 7","Арал плюс, Катрен г.Химки, Протек, Пульс","Юник Фармасьютикал Лабораториз (Отделение Дж.Б. Ке, Юник Фармасьютикал Лабораториз/Дж Б Кемикалс Фарма",179.59,208.62,247.0,293.0


Строки по датам и аптекам:


,dr_dat,shop,dr_cdrugs,n_rows,qty,zak,roz
0,2022-05-05,Аптека 11,336283,1,1.0,182.18,250.0
1,2022-05-19,Аптека 2,28901,1,1.0,180.69,255.0
2,2022-05-24,Аптека 2,28901,2,1.0,180.69,255.0
3,2022-05-26,Аптека 11,28901,1,1.0,179.59,247.0
4,2022-05-26,Аптека 7,28901,1,1.0,179.59,250.0
5,2022-05-29,Аптека 3,28901,1,1.0,208.62,293.0
6,2022-05-30,Аптека 7,28901,1,1.0,181.76,253.0
7,2022-06-04,Аптека 11,28901,1,1.0,182.18,250.0
8,2022-06-08,Аптека 10,28901,1,1.0,183.58,248.0
9,2022-06-08,Аптека 11,28901,1,1.0,179.59,247.0



ТОВАР: ПЕРЕКИСЬ ВОДОРОДА 3% 100МЛ. Р-Р ФЛ. ПЛАСТ. /ТУЛЬСКАЯ ФФ/


,dr_cdrugs,n_rows,first_date,last_date,shops,suppliers,producers,zak_min,zak_max,roz_min,roz_max
1,1504470,1,2022-05-03,2022-05-03,Аптека 3,Магнит Фарма ООО,Гедеон Рихтер А.О,46.68,46.68,65.0,65.0
0,346681,84,2022-05-04,2022-06-07,"Аптека 11, Аптека 2, Аптека 4, Аптека 7","Магнит Фарма ООО, ПрофитМед",ТУЛЬСКАЯ ФАРМ. ФАБРИКА,9.46,10.34,11.0,12.0


Строки по датам и аптекам:


,dr_dat,shop,dr_cdrugs,n_rows,qty,zak,roz
0,2022-05-03,Аптека 3,1504470,1,1.0,46.68,65.0
1,2022-05-04,Аптека 11,346681,2,2.0,9.46,11.0
2,2022-05-05,Аптека 11,346681,1,1.0,9.46,11.0
3,2022-05-05,Аптека 2,346681,7,7.0,10.34,12.0
4,2022-05-06,Аптека 11,346681,10,10.0,9.46,11.0
5,2022-05-07,Аптека 11,346681,1,1.0,9.46,11.0
6,2022-05-08,Аптека 11,346681,2,2.0,9.46,11.0
7,2022-05-10,Аптека 11,346681,1,1.0,9.46,11.0
8,2022-05-10,Аптека 2,346681,2,2.0,10.34,12.0
9,2022-05-10,Аптека 7,346681,2,2.0,10.34,12.0


- Для ДОКТОР МОМ разовый артикул 336283 был только 2022-05-05 в Аптека 11, а основной 28901 дальше во всех случаях.
- Для ПЕРЕКИСЬ ВОДОРОДА аналогично, разовый артикул 1504470 был только 2022-05-03 в Аптека 3, а основной 346681 потом используется дальше.
- Такой паттерн очень похож на ошибку первичного заведения товара или на временную карточку, которую затем заменили на правильную.  

**❗ Решение: объединить эти пары SKU в один основной артикул**

In [26]:
# Проблемные артикулы: одному артикулу соответствует больше 1 названия
problem_skus = (
    sales_final.groupby('dr_cdrugs')['dr_ndrugs']
    .nunique()
    .reset_index(name='n_names')
    .query('n_names > 1')['dr_cdrugs']
    .tolist()
)

for sku in problem_skus:
    print(f"\n{'='*80}")
    print(f"АРТИКУЛ: {sku}")
    print(f"{'='*80}")

    tmp = sales_final[sales_final['dr_cdrugs'] == sku].copy()

    # Сводка по названиям
    summary = (
        tmp.groupby('dr_ndrugs')
        .agg(
            n_rows=('dr_ndrugs', 'size'),
            first_date=('dr_dat', 'min'),
            last_date=('dr_dat', 'max'),
            shops=('shop', lambda x: ', '.join(sorted(x.astype(str).unique()))),
            suppliers=('dr_suppl', lambda x: ', '.join(sorted(x.dropna().astype(str).unique()))),
            producers=('dr_prod', lambda x: ', '.join(sorted(x.dropna().astype(str).unique()))),
            zak_min=('dr_czak', 'min'),
            zak_max=('dr_czak', 'max'),
            roz_min=('dr_croz', 'min'),
            roz_max=('dr_croz', 'max')
        )
        .reset_index()
        .sort_values(['first_date', 'dr_ndrugs'])
    )

    display(summary)


АРТИКУЛ: 816


,dr_ndrugs,n_rows,first_date,last_date,shops,suppliers,producers,zak_min,zak_max,roz_min,roz_max
1,НИТРОФУНГИН-ТЕВА 25МЛ. №1 СПИРТ. Р-Р ФЛ.,1,2022-05-20,2022-05-20,Аптека 4,Протек,ТЕВА ФАРМАЦЕВТИЧЕСКИЕ ПРЕДПРИЯТИЯ ЛТД,378.43,378.43,523.0,523.0
0,НИТРОФУНГИН-ТЕВА 1% 25МЛ. №1 СПИРТ. Р-Р ФЛ.,5,2022-05-26,2022-06-05,"Аптека 2, Аптека 3, Аптека 4, Аптека 7","Протек, Пульс","TEVA, ТЕВА ФАРМАЦЕВТИЧЕСКИЕ ПРЕДПРИЯТИЯ ЛТД",318.36,390.58,405.0,528.0



АРТИКУЛ: 1247


,dr_ndrugs,n_rows,first_date,last_date,shops,suppliers,producers,zak_min,zak_max,roz_min,roz_max
1,ТРИАМПУР КОМПОЗИТУМ №50 ТАБ.,1,2022-05-16,2022-05-16,Аптека 11,Катрен г.Химки,Плива Хрватска д.о.о.,354.89,354.89,462.0,462.0
0,"ТРИАМПУР КОМПОЗИТУМ 12,5МГ+25МГ. №50 ТАБ.",1,2022-06-02,2022-06-02,Аптека 7,ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК,Плива Хрватска д.о.о.,431.67,431.67,583.0,583.0



АРТИКУЛ: 2468


...

,dr_ndrugs,n_rows,first_date,last_date,shops,suppliers,producers,zak_min,zak_max,roz_min,roz_max
1,ДИФЕРТОН №60 ТАБ. П/О,1,2022-05-07,2022-05-07,Аптека 7,Протек,ВТФ ООО,956.34,956.34,1406.0,1406.0
0,"ДИФЕРТОН 1,43Г. №60 ТАБ. П/О",1,2022-05-27,2022-05-27,Аптека 2,Протек,ВТФ ООО,991.66,991.66,1438.0,1438.0



АРТИКУЛ: 561002


,dr_ndrugs,n_rows,first_date,last_date,shops,suppliers,producers,zak_min,zak_max,roz_min,roz_max
0,ЭВО КРЕМ Д/РУК МОЧЕВИНА 100МЛ. [EVO] /АВАНТА/,2,2022-05-14,2022-06-02,"Аптека 2, Аптека 8",Катрен г.Химки,АВАНТА ОАО,83.60,92.15,133.0,145.0
1,ЭВО КРЕМ Д/РУК МОЧЕВИНА 100МЛ. ТУБА [EVO] /АВАНТА/,1,2022-06-08,2022-06-08,Аптека 8,Катрен г.Химки,АВАНТА ОАО,80.12,80.12,126.0,126.0



АРТИКУЛ: 595888


,dr_ndrugs,n_rows,first_date,last_date,shops,suppliers,producers,zak_min,zak_max,roz_min,roz_max
1,НИКОФЛЕКС КРЕМ 50Г. ТУБА,3,2022-05-08,2022-05-19,"Аптека 10, Аптека 2",Катрен г.Химки,МЕДИМПЕКС ЗАО,214.84,216.60,312.0,334.0
0,НИКОФЛЕКС 50Г. КРЕМ Д/НАРУЖ. ПРИМ. ТУБА,1,2022-05-26,2022-05-26,Аптека 10,Катрен г.Химки,МЕДИМПЕКС ЗАО,211.02,211.02,306.0,306.0


- В данных встречаются вариации названий одного и того же товара, связанные с указанием поставщика, формы выпуска и других уточнений.
- **❗ Поэтому в товарном анализе в качестве основного ключа используется артикул (`dr_cdrugs`), а название оставляется как текстовое описание товара**.

In [27]:
# Замена ошибочных артикулов на основные
sku_map = {
    336283: 28901,   # ДОКТОР МОМ ФИТО 20Г. МАЗЬ Д/НАРУЖ.ПРИМ. БАНКА
    1504470: 346681  # ПЕРЕКИСЬ ВОДОРОДА 3% 100МЛ. Р-Р ФЛ. ПЛАСТ. /ТУЛЬСКАЯ ФФ/
}

sales_final = sales_final.copy()
sales_final['dr_cdrugs'] = sales_final['dr_cdrugs'].replace(sku_map)

### 🕵🏻 Наблюдение 4
- Были выявлены неоднозначности: 60 артикулов с более чем одним названием и 2 названия с 2-мя артикулами.
- По товарам Доктор Мом и Перекись водорода (одно название - 2 артикула)установили, что один из артикулов в каждой паре был 1 раз, а второй во всех остальных случаях. Решение: заменили разовые артикулы на основные.
- Что касается обратной ситуации одному артикулу соответсвует несколько названий, установили, что это вариации названий одного и того же товара, связанные с указанием поставщика, формы выпуска и других уточнений. Решение: использовать `dr_cdrugs` как основной идентификатор товара, а `dr_ndrugs` - как описательное поле.

## Этап 2. Распределения в данных

### Уникальные значения

In [28]:
print("Уникальные значения для категориальных признаков:")
for col in ['dr_tdoc', 'dr_tpay', 'dr_suppl', 'dr_cdisc', 'shop']:
    print(f"\n{col}: {sales_final[col].unique()}")

print("\nКоличество уникальных значений для идентификаторов:")
for col in ['dr_nchk', 'check_id', 'dr_ndoc', 'dr_kkm', 'dr_cdrugs', 'dr_bcdisc', 'dr_prod']:
    print(f"{col}: {sales_final[col].nunique()} уникальных значений")

Уникальные значения для категориальных признаков:

dr_tdoc: ['Розничная реализация']

dr_tpay: [18 15]

dr_suppl: ['Фармкомплект ООО' 'Катрен г.Химки' 'Авеста'
 'ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК' 'ЕАПТЕКА ООО' 'Протек' 'Пульс'
 'Здравсервис' 'ГК Надежда Фарм' 'Арал плюс' 'ПрофитМед'
 'ВИТТА КОМПАНИ ООО' 'Норман' 'ООО ИДВ' 'СТЭЛМАС-Д ООО' 'БСС'
 'Магнит Фарма ООО' 'Индивидуальный предприниматель Кочанов Андрей Миха'
 'ООО "Акцентмед"' 'Подотчетное лицо' 'СиЭс Медика Калуга' 'ООО "КОМУС"'
 'Вернигор Николай Викторович' 'АЛВИЛС ООО' 'ОДАС ФАРМА ООО'
 'АйТи-Аптека Внедрение']

dr_cdisc: <IntegerArray>
[9, 35, <NA>, 925, 30, 941, 27, 939, 11, 7, 28, 37]
Length: 12, dtype: Int64

shop: ['Аптека 4' 'Аптека 3' 'Аптека 2' 'Аптека 8' 'Аптека 7' 'Аптека 11'
 'Аптека 10' 'Аптека 1']

Количество уникальных значений для идентификаторов:
dr_nchk: 4999 уникальных значений
check_id: 20932 уникальных значений
dr_ndoc: 399 уникальных значений
dr_kkm: 12 уникальных значений
dr_cdrugs: 6497 уникальных значени

### 🕵🏻 Наблюдение 5
- Все записи относятся к Розничная реализация, что означает, что в данных представлены только розничные продажи.
- Присутствуют два типа оплаты: 18 и 15 (безналичная и наличная оплата, как упоминалось ранее в описании).
- Обнаружено большое количество уникальных поставщиков и производителей, что указывает на широкое разнообразие источников товаров.
- В данных встречается 11 кодов скидок и отсутствие скидки (NA), код скидки встречается у большого числа строк. Чтобы сделать более точный вывод, нужно смотреть распределение `dr_bcdisc` по `dr_cdisc`.
- Номеров чеков ожидаемо меньше, чем уникальных чеков (комбинация `dr_apt`, `dr_nchk` и `dr_dat`), т.к. номер чека может повторятся в разных аптеках в разные даты.
- 12 уникальных кассовых аппаратов на 8 точек продаж (аптек).
- 6 497 унакальных товара продано за период во всех аптеках (по артикулу).

### Бонусные карты

In [29]:
# Смотрим, сколько штрихкодов на каждый тип скидки
unique_bcdisc_per_cdisc = sales_final.groupby('dr_cdisc')['dr_bcdisc'].nunique().reset_index()
unique_bcdisc_per_cdisc.columns = ['dr_cdisc', 'unique_dr_bcdisc_count']
display(unique_bcdisc_per_cdisc)

,dr_cdisc,unique_dr_bcdisc_count
0,7,1
1,9,2271
2,11,8
3,27,1
4,28,4
5,30,1
6,35,1
7,37,1
8,925,1
9,939,1


- Есть варианты, когда скидка не применялась - это пропуски в `dr_cdisc` и `dr_bcdisc`.
- Есть варианты `dr_cdisc` = 9, 11, 28 - это когда применялась скидка по карте лояльности:  
| Код скидки | Название |
|---|---|
| 9 | Пенсионный дисконт |
| 11 | VIP |
| 28 | VIP2 |  
- Остальные (7, 27, 30, 35, 37, 925, 939, 941) - это просто прошла скидка (техническая, карты лояльности не было):  
| Код скидки | Штрихкод скидки | Название |
|---|---|---|
| 7 | 200000000006 | Скидка 15% |
| 27 | ... | Сотрудники АЦ |
| 30 | 200000000024 | Акции |
| 35 | 200000000022 | Аптека №3 |
| 37 | 200000000026 | Стол заказов |
| 925 | 200000000492 | Рарус |
| 939 | 200000000042 | Скидок по срокам годности |
| 941 | 200000000044 | Профком |

In [30]:
disc_df = sales_final.copy()

loyalty_codes = [9, 11, 28]
tech_codes = [7, 27, 30, 35, 37, 925, 939, 941]

discount_name_map = {
    9: 'Пенсионный дисконт',
    11: 'VIP',
    28: 'VIP2',
    7: 'Скидка 15%',
    27: 'Сотрудники АЦ',
    30: 'Акции',
    35: 'Аптека №3',
    37: 'Стол заказов',
    925: 'Рарус',
    939: 'Скидок по срокам годности',
    941: 'Профком'
}

disc_df['discount_type'] = 'other_discount'
disc_df.loc[disc_df['dr_cdisc'].isna(), 'discount_type'] = 'no_discount'
disc_df.loc[disc_df['dr_cdisc'].isin(loyalty_codes), 'discount_type'] = 'loyalty_card'
disc_df.loc[disc_df['dr_cdisc'].isin(tech_codes), 'discount_type'] = 'technical_discount'

disc_df['discount_name'] = disc_df['dr_cdisc'].map(discount_name_map)

pension_discount = disc_df[disc_df['dr_cdisc'] == 9].copy()

cdisc_per_check_all = disc_df.groupby('check_id')['dr_cdisc'].nunique(dropna=True)
multiple_cdisc_checks_all = cdisc_per_check_all[cdisc_per_check_all > 1]
print(f"Количество check_id с разными dr_cdisc: {len(multiple_cdisc_checks_all)}")
if not multiple_cdisc_checks_all.empty:
    display(multiple_cdisc_checks_all.head(10))

multiple_cdisc_checks_loyalty = (
    disc_df.loc[disc_df['dr_cdisc'].isin(loyalty_codes)]
    .groupby('check_id')['dr_cdisc']
    .nunique()
)
multiple_cdisc_checks_loyalty = multiple_cdisc_checks_loyalty[multiple_cdisc_checks_loyalty > 1]
print(f"Количество check_id с разными loyalty dr_cdisc (9, 11, 28): {len(multiple_cdisc_checks_loyalty)}")
if not multiple_cdisc_checks_loyalty.empty:
    display(multiple_cdisc_checks_loyalty.head(10))

check_discount = (
    disc_df.groupby('check_id', as_index=False)
    .agg(
        shop=('shop', 'first'),
        has_discount=('dr_cdisc', lambda s: s.notna().any()),
        has_loyalty=('dr_cdisc', lambda s: s.isin(loyalty_codes).any()),
        has_tech_discount=('dr_cdisc', lambda s: s.isin(tech_codes).any()),
        max_disc_sum=('dr_sdisc', 'max'),
        total_disc_sum=('dr_sdisc', 'sum')
    )
)

check_discount['has_discount'] = check_discount['has_discount'].fillna(False).astype(bool)
check_discount['has_loyalty'] = check_discount['has_loyalty'].fillna(False).astype(bool)
check_discount['has_tech_discount'] = check_discount['has_tech_discount'].fillna(False).astype(bool)

check_discount['check_group'] = 'other'
check_discount.loc[~check_discount['has_discount'], 'check_group'] = 'no_discount'
check_discount.loc[
    check_discount['has_loyalty'] & ~check_discount['has_tech_discount'],
    'check_group'
] = 'loyalty_only'
check_discount.loc[
    ~check_discount['has_loyalty'] & check_discount['has_tech_discount'],
    'check_group'
] = 'technical_only'
check_discount.loc[
    check_discount['has_loyalty'] & check_discount['has_tech_discount'],
    'check_group'
] = 'mixed_loyalty_and_tech'

check_summary = (
    check_discount.groupby('check_group', as_index=False)
    .agg(
        checks=('check_id', 'size'),
        shops=('shop', 'nunique'),
        avg_max_disc_sum=('max_disc_sum', 'mean'),
        med_max_disc_sum=('max_disc_sum', 'median'),
        avg_total_disc_sum=('total_disc_sum', 'mean'),
        med_total_disc_sum=('total_disc_sum', 'median')
    )
)

check_summary['share_checks'] = check_summary['checks'] / check_summary['checks'].sum()
display(check_summary.sort_values('checks', ascending=False))

row_summary = (
    disc_df.groupby('discount_type', as_index=False)
    .agg(
        rows=('check_id', 'size'),
        checks=('check_id', 'nunique')
    )
)

row_summary['share_rows'] = row_summary['rows'] / row_summary['rows'].sum()
row_summary['share_checks'] = row_summary['checks'] / row_summary['checks'].sum()
display(row_summary.sort_values('rows', ascending=False))

final_discount_report = pd.DataFrame([{
    'total_rows': len(disc_df),
    'total_checks': disc_df['check_id'].nunique(),
    'no_discount_checks': int((check_discount['check_group'] == 'no_discount').sum()),
    'loyalty_only_checks': int((check_discount['check_group'] == 'loyalty_only').sum()),
    'technical_only_checks': int((check_discount['check_group'] == 'technical_only').sum()),
    'mixed_checks': int((check_discount['check_group'] == 'mixed_loyalty_and_tech').sum()),
    'pension_checks': pension_discount['check_id'].nunique(),
}])

display(final_discount_report)

Количество check_id с разными dr_cdisc: 44


,dr_cdisc
check_id,
11_1363_20220523,2
11_501_20220518,2
11_8197_20220504,2
11_8699_20220507,2
13_1731_20220508,2
13_1773_20220514,2
13_1903_20220515,2
13_3941_20220506,2
13_5027_20220525,2


Количество check_id с разными loyalty dr_cdisc (9, 11, 28): 0


,check_group,checks,shops,avg_max_disc_sum,med_max_disc_sum,avg_total_disc_sum,med_total_disc_sum,share_checks
2,no_discount,13335,8,0.000000,0.0,0.000000,0.0,0.637063
0,loyalty_only,3859,8,26.567225,16.0,40.112633,22.0,0.184359
3,technical_only,3721,8,53.169221,33.0,73.920253,43.0,0.177766
1,mixed_loyalty_and_tech,17,6,100.152941,47.0,172.817647,93.0,0.000812


,discount_type,rows,checks,share_rows,share_checks
1,no_discount,26867,14143,0.59967,0.650044
0,loyalty_card,9733,3876,0.21724,0.178150
2,technical_discount,8203,3738,0.18309,0.171807


,total_rows,total_checks,no_discount_checks,loyalty_only_checks,technical_only_checks,mixed_checks,pension_checks
0,44803,20932,13335,3859,3721,17,3845


In [31]:
# Посмотрим пенсионные скидки отдельно
pention_rows = sales_final[sales_final['dr_cdisc'] == 9].copy()
pention_rows['row_revenue'] = pention_rows['dr_kol'] * pention_rows['dr_croz']

pention_check = (
    pention_rows.groupby('check_id', as_index=False)
    .agg(
        shop=('shop', 'first'),
        rows=('check_id', 'size'),
        total_qty=('dr_kol', 'sum'),
        total_revenue=('row_revenue', 'sum'),
        total_disc_sum=('dr_sdisc', 'sum'),
        max_disc_sum=('dr_sdisc', 'max'),
        mean_roz=('dr_croz', 'mean'),
        mean_zak=('dr_czak', 'mean')
    )
)

display(pention_check.describe())

,rows,total_qty,total_revenue,total_disc_sum,max_disc_sum,mean_roz,mean_zak
count,3845.000000,3845.000000,3845.000000,3845.000000,3845.000000,3845.000000,3845.000000
mean,2.512614,2.667793,778.876965,39.633012,26.196645,382.214199,300.224938
std,2.183244,2.693955,1115.182250,56.644277,32.181101,506.427501,425.187278
min,1.000000,0.020000,16.000000,0.500000,0.500000,3.200000,1.980000
25%,1.000000,1.000000,189.000000,9.000000,6.000000,107.666667,80.300000
50%,2.000000,2.000000,440.000000,22.000000,16.000000,231.666667,176.260000
75%,3.000000,3.000000,933.000000,50.000000,34.000000,477.000000,370.150000
max,30.000000,41.000000,33739.000000,1681.000000,507.000000,12316.000000,11165.000000


### 🕵🏻 Наблюдение 6
- Карта лояльности выделяется как отдельный тип скидки, а технические скидки живут отдельно и лишь в редких случаях смешиваются в одном чеке. Это хорошо подтверждается тем, что для кодов 9, 11, 28 не найдено чеков с несколькими разными значениями `dr_cdisc`, тогда как в целом по базе такие смешанные чеки есть.
- Большинство чеков проходит без скидки - 13 335 (64%).
- Чеки с картой лояльности (loyalty_only) - 3 859 (18%).
- Технические скидки (technical_only, не карта лояльности) - 3 721 (тоже 18%). Такие скидки используются как операционный механизм (скидки по акции, сроку годности, внутренние скидки и т.д.).
- Встречается 17 чеков, где с картой лояльности использовалась техническая скидка. Если в чеке встретилась лояльность (9/11/28), она почти всегда идет без смешения с другими типами скидок, и это делает интерпретацию этих кодов достаточно надежной.
- Пенсионный дисконт (тип 9) является наиболее массовым среди клиентских скидок и охватывает значимое число чеков. При этом средние значения выручки, количества позиций и суммы скидки указывают на то, что он применяется в обычных покупательских сценариях. Распределение имеет выраженный правый хвост, что говорит о наличии отдельных крупных чеков.

### Типы оплат

In [32]:
# Доля по типам оплаты
payment_type_proportion = sales_final['dr_tpay'].value_counts(normalize=True) * 100.00
print("Доля от общего числа по типам оплаты (%):")
display(payment_type_proportion.round(2))

Доля от общего числа по типам оплаты (%):


,proportion
dr_tpay,
18,61.1
15,38.9


In [33]:
# Доля по типам оплаты в каждой аптеке
payment_type_by_shop = sales_final.groupby('shop')['dr_tpay'].value_counts(normalize=True).unstack(fill_value=0) * 100
print("Доля от общего числа по типам оплаты (%) в каждой аптеке:")
display(payment_type_by_shop.round(2))

Доля от общего числа по типам оплаты (%) в каждой аптеке:


dr_tpay,15,18
shop,,
Аптека 1,51.31,48.69
Аптека 10,16.79,83.21
Аптека 11,32.60,67.40
Аптека 2,45.03,54.97
Аптека 3,35.88,64.12
Аптека 4,54.44,45.56
Аптека 7,36.22,63.78
Аптека 8,44.47,55.53


In [34]:
# Проверка на уникальность dr_tpay для каждого check_id
tpay_check = sales_final.groupby('check_id')['dr_tpay'].nunique()
multiple_tpay_checks = tpay_check[tpay_check > 1]
display(multiple_tpay_checks.head())
print(f"Кол-во check_id с 2-мя типами оплат: {len(multiple_tpay_checks)}")

,dr_tpay
check_id,
11_1349_20220523,2
11_2075_20220527,2
11_2627_20220531,2
11_2723_20220531,2
11_3055_20220603,2


Кол-во check_id с 2-мя типами оплат: 81


### 🕵🏻 Наблюдение 7
- 61% покупателей платит безналично, 39% - наличными.
- В разрезе аптек наличная оплата преобладает только в Аптеках 1 и 4 (ближе к 50/50).
- В Аптеках 2 и 8 безналичная оплата незначительно преобладает, в Аптеках 3 и 7 преобладает более значительно, а в Аптеках 10 и 11 - доминирует.
- Скорее всего существует возможность комбинированной оплаты (гипотеза). Некоторые чеки были оплачены частично наличными (тип 15) и частично картой (тип 18). Система зафиксировала обе операции в рамках одного чека.

### Доля интернет-заказов

In [35]:
# Доля по типам заказов (dr_vzak: 1 - обычный, 2 - интернет-заказ)
order_type_proportion = sales_final['dr_vzak'].value_counts(normalize=True) * 100
print("Доля от общего числа по типам заказов (%):")
display(order_type_proportion.round(2))

Доля от общего числа по типам заказов (%):


,proportion
dr_vzak,
1,95.46
2,4.54


In [36]:
internet_orders_by_shop = sales_final.groupby('shop')['dr_vzak'].value_counts(normalize=True).unstack(fill_value=0) * 100
internet_orders_by_shop = internet_orders_by_shop.rename(columns={1: 'заказ в реале', 2: 'интернет-заказ'})

print("Доля обычных и интернет-заказов (%) в каждой аптеке:")
display(internet_orders_by_shop.round(2))

Доля обычных и интернет-заказов (%) в каждой аптеке:


dr_vzak,заказ в реале,интернет-заказ
shop,,
Аптека 1,98.16,1.84
Аптека 10,93.44,6.56
Аптека 11,100.00,0.00
Аптека 2,96.02,3.98
Аптека 3,95.25,4.75
Аптека 4,95.57,4.43
Аптека 7,88.18,11.82
Аптека 8,95.36,4.64


### 🕵🏻 Наблюдение 8
- В основном покупки совершаются в реале. Доля интернет-заказов - всего 4.5%.
- В Аптеке 11 вообще не зафиксированы интернет-заказы.
- Наибольшее количество интернет-заказов в Аптеке 7 (~12%). В остальных случаях - от 1.9% до 6.5%.

### Аномалии в датах

In [37]:
df = sales_final.copy()
# df.info()

In [38]:
# Дата
df['date'] = df['dr_dat'].dt.date
df['weekday'] = df['dr_dat'].dt.day_name()

# Время
df['dr_tim'] = pd.to_datetime(df['dr_tim'].astype(str), format='%H:%M:%S', errors='coerce')
df['hour'] = df['dr_tim'].dt.hour

In [39]:
# Диапазон дат и отсутствующие даты
full_dates = pd.date_range(df['dr_dat'].min().normalize(), df['dr_dat'].max().normalize(), freq='D')
present_dates = pd.to_datetime(df['date'].dropna().unique())
missing_dates = full_dates.difference(present_dates)

print("Мин. дата:", df['dr_dat'].min())
print("Макс. дата:", df['dr_dat'].max())
print("Уникальных дат:", df['date'].nunique())
print("Отсутствующих дат:", len(missing_dates))

display(pd.DataFrame({'missing_dates': missing_dates}))

Мин. дата: 2022-05-01 00:00:00
Макс. дата: 2022-06-09 00:00:00
Уникальных дат: 39
Отсутствующих дат: 1


,missing_dates
0,2022-05-09


In [40]:
# Продажи по дням недели
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

df = sales_final.copy()
df['weekday'] = df['dr_dat'].dt.day_name()
df['row_revenue'] = df['dr_croz'] * df['dr_kol']

weekday_stats = (
    df.groupby('weekday', as_index=False)
    .agg(
        rows=('dr_cdrugs', 'size'), # интенсивность присутствия товара
        checks=('check_id', 'nunique'), # частота покупок
        qty=('dr_kol', 'sum'), # реальный объем продаж
        revenue=('row_revenue', 'sum') # денежный вклад
    )
    .set_index('weekday')
    .reindex(weekday_order)
    .reset_index()
)

weekday_stats['qty'] = weekday_stats['qty'].round(0).astype('Int64')
weekday_stats['revenue'] = weekday_stats['revenue'].round(0).astype('Int64')

display(weekday_stats)

,weekday,rows,checks,qty,revenue
0,Monday,6045,2804,6338,1672952
1,Tuesday,7003,3310,7594,1955024
2,Wednesday,7243,3393,8158,2189303
3,Thursday,6984,3248,7480,1979343
4,Friday,6256,2825,6662,1752674
5,Saturday,5501,2532,5943,1455581
6,Sunday,5771,2820,6132,1629175


In [41]:
# Продажи по дням недели по аптекам
weekday_shop_stats = (
    df.groupby(['shop', 'weekday'], as_index=False)
    .agg(
        rows=('dr_cdrugs', 'size'),
        checks=('check_id', 'nunique'),
        qty=('dr_kol', 'sum'),
        revenue=('row_revenue', 'sum')
    )
)

weekday_shop_stats['weekday'] = pd.Categorical(
    weekday_shop_stats['weekday'],
    categories=weekday_order,
    ordered=True
)

weekday_shop_stats = weekday_shop_stats.sort_values(['shop', 'weekday'])

weekday_shop_stats['qty'] = weekday_shop_stats['qty'].round(0).astype('Int64')
weekday_shop_stats['revenue'] = weekday_shop_stats['revenue'].round(0).astype('Int64')

display(weekday_shop_stats)

,shop,weekday,rows,checks,qty,revenue
1,Аптека 1,Monday,716,305,773,183727
5,Аптека 1,Tuesday,810,334,937,214380
6,Аптека 1,Wednesday,861,365,913,228108
4,Аптека 1,Thursday,851,362,947,218653
0,Аптека 1,Friday,849,342,903,202562
2,Аптека 1,Saturday,605,233,663,133700
3,Аптека 1,Sunday,539,211,555,128442
8,Аптека 10,Monday,580,300,572,162595
12,Аптека 10,Tuesday,715,357,762,217202
13,Аптека 10,Wednesday,747,369,793,252004


In [42]:
# Визулизация дней недели по аптекам (выручка)
plot_df = weekday_shop_stats.copy()
plot_df['weekday'] = pd.Categorical(plot_df['weekday'], categories=weekday_order, ordered=True)
plot_df = plot_df.sort_values(['shop', 'weekday'])

fig = px.line(
    plot_df,
    x='weekday',
    y='revenue',
    color='shop',
    markers=True,
    category_orders={'weekday': weekday_order},
    title='Выручка по дням недели в разрезе аптек'
)

fig.update_layout(
    legend_title_text='Аптека'
)
fig.update_xaxes(title_text='День недели')
fig.update_yaxes(title_text='Выручка')
fig.show()

In [43]:
# Визулизация дней недели по аптекам (кол-во чеков)
plot_df = weekday_shop_stats.copy()
plot_df['weekday'] = pd.Categorical(plot_df['weekday'], categories=weekday_order, ordered=True)
plot_df = plot_df.sort_values(['shop', 'weekday'])

fig = px.line(
    plot_df,
    x='weekday',
    y='checks',
    color='shop',
    markers=True,
    category_orders={'weekday': weekday_order},
    title='Кол-во чеков по дням недели в разрезе аптек'
)

fig.update_layout(
    legend_title_text='Аптека'
)
fig.update_xaxes(title_text='День недели')
fig.update_yaxes(title_text='Кол-во чеков')
fig.show()

### 🕵🏻 Наблюдение 9
- В данных наблюдается почти непрерывный календарный ряд: период с 1 мая 2022 года по 9 июня 2022 года содержит 39 уникальных дат, при этом отсутствует только 9 мая 2022 года. Возможно в праздник аптечная сеть не работала.
- Наибольшая активность приходится на среду, затем идут вторник и четверг (выручка и кол-во чеков). Минимальные значения наблюдаются в субботу, что похоже на более низкий поток покупателей в выходные, но в воскресение уже начинает расти.
- Аптеки:
  - Аптека 1. Пик продаж приходится на вторник-четверг. По количеству чеков и по выручке выходные заметно слабее будней, при этом падение не резкое, а умеренное. Аптека выглядит как точка с относительно стабильным спросом в течение недели.
  - Аптека 2. Продажи высокие всю неделю, с выраженным пиком в середине недели. Число чеков и выручка снижаются к выходным, но точка остаётся активной даже в субботу и воскресенье. Это похоже на крупную и стабильную аптеку с ровным недельным профилем.
  - Аптека 3. Выручка и число чеков распределены довольно равномерно, с небольшим усилением во вторник и среду. Выходные ниже будней, но без резкого провала. Аптека выглядит как средняя по масштабу точка с нормальной недельной сезонностью.
  - Аптека 4. Это самая спокойная точка: и выручка, и число чеков здесь заметно ниже, чем у остальных. Профиль по дням недели довольно ровный, без сильных пиков. По ощущению это формат "у дома" или небольшая точка с более скромным трафиком.
  - Аптека 7. Хорошо выражен срединный пик недели, особенно во вторник-четверг. В выходные активность падает, но точка остается достаточно сильной и по чекам, и по выручке. Аптека выглядит крупной и достаточно динамичной.
  - Аптека 8. По количеству чеков и выручке точка средняя, но с очень заметной концентрацией на середине недели. В выходные активность чуть ниже, чем в будни. В целом это стабильная аптека без резких провалов.
  - Аптека 10. Основная активность сосредоточена в будни, особенно во вторник-четверг. В воскресенье и субботу продажи ниже, но спад не критичный. в четверг видно расхождение между количеством чеков и выручкой: чеков становится больше, но выручка снижается. Это может означать, что в этот день растёт доля более мелких покупок или снижается средний чек.
  - Аптека 11. Это одна из самых активных точек по абсолютным значениям: и выручка, и число чеков выше, чем у большинства других аптек. Пик приходится на среду и воскресенье, что выделяет ее среди остальных. Аптека выглядит как крупная точка с устойчивым потоком покупателей.
- Аптеки 2 и 11 стабильно входят в лидирующую группу, однако их позиции меняются местами в зависимости от метрики: одна сильнее по выручке, другая - по количеству чеков. Это указывает на различия в среднем чеке и структуре покупательского спроса.

### Пиковые часы

In [44]:
df_hours = df.copy()

# переводим строку времени в datetime
df_hours['dr_tim'] = pd.to_datetime(df_hours['dr_tim'], format='%H:%M:%S', errors='coerce')

# достаём час
df_hours['hour'] = df_hours['dr_tim'].dt.hour

hourly_shop_stats = (
    df_hours.groupby(['shop', 'hour'], as_index=False)
    .agg(
        checks=('check_id', 'nunique'),
        revenue=('row_revenue', 'sum')
    )
    .sort_values(['shop', 'hour'])
)

display(hourly_shop_stats.head())
print(f'Диапазон часов: {hourly_shop_stats['hour'].min()} - {hourly_shop_stats['hour'].max()}')

,shop,hour,checks,revenue
0,Аптека 1,7,1,89.000000
1,Аптека 1,8,73,47160.078804
2,Аптека 1,9,120,67656.363291
3,Аптека 1,10,209,139664.842301
4,Аптека 1,11,237,159582.611915


Диапазон часов: 7 - 22


In [45]:
# График по чекам
fig = px.line(
    hourly_shop_stats,
    x='hour',
    y='checks',
    color='shop',
    markers=True,
    title='Чеки по часам в разрезе аптек'
)

fig.update_xaxes(title_text='Час')
fig.update_yaxes(title_text='Чеки')
fig.update_layout(legend_title_text='Аптека')
fig.show()

In [46]:
# График по выручке
fig = px.line(
    hourly_shop_stats,
    x='hour',
    y='revenue',
    color='shop',
    markers=True,
    title='Выручка по часам в разрезе аптек'
)

fig.update_xaxes(title_text='Час')
fig.update_yaxes(title_text='Выручка')
fig.update_layout(legend_title_text='Аптека')
fig.show()

### 🕵🏻 Наблюдение 10
- Судя по всему, аптеки работают с 7.00 до 22.00.
- Существенный рост количества чеков и выручки наблюдается с 8-9 утра, достигая мини-пика к 10.00-11.00.
- Далее до 17.00 относительно стабильный и высокий уровень. Покупатели приходят в течение рабочего дня.
- Также активность по выручке и чекам есть вечером 17.00-19.00. Характерно для покупок после работы.
- После 19.00 активность быстро снижается.
- Аптека 10 имеет вечерний пик, начинающийся чуть раньше и сохраняющийся дольше, чем у других аптек. Это может указывать на расположение в районе с большим количеством офисов или транспортным узлом.
- В целом аптеки сети следуют общей тенденции.

## Этап 3. Товарный анализ

### Несколько значений цены
На практике цены в аптеке часто меняются и между точками, и во времени, особенно из-за новых поставок и обновления остатков.
- сколько товаров с одной/несколькими закупочными и розничными ценами
- отдельно в разрезе аптек
- отдельно по времени

In [47]:
df = sales_final.copy()

price_summary = (
df.groupby('dr_cdrugs', as_index=False)
.agg(
dr_ndrugs=('dr_ndrugs', 'first'),
zak_unique=('dr_czak', 'nunique'),
roz_unique=('dr_croz', 'nunique'),
rows=('dr_cdrugs', 'size')
))

print("=== По всем данным ===")
print(f"Всего уникальных товаров: {df['dr_cdrugs'].nunique()}")
print(f"Товаров с 1 закупочной ценой: {(price_summary['zak_unique'] == 1).sum()}")
print(f"Товаров с >1 закупочной ценой: {(price_summary['zak_unique'] > 1).sum()}")
print(f"Товаров с 1 розничной ценой: {(price_summary['roz_unique'] == 1).sum()}")
print(f"Товаров с >1 розничной ценой: {(price_summary['roz_unique'] > 1).sum()}")

# по аптекам
shop_price_summary = (
df.groupby(['shop', 'dr_cdrugs'], as_index=False)
.agg(
dr_ndrugs=('dr_ndrugs', 'first'),
zak_unique=('dr_czak', 'nunique'),
roz_unique=('dr_croz', 'nunique'),
rows=('dr_cdrugs', 'size')
))

shop_counts = (
shop_price_summary.groupby('shop', as_index=False)
.agg(
sku_count=('dr_cdrugs', 'nunique'),
zak_fixed=('zak_unique', lambda s: (s == 1).sum()),
zak_var=('zak_unique', lambda s: (s > 1).sum()),
roz_fixed=('roz_unique', lambda s: (s == 1).sum()),
roz_var=('roz_unique', lambda s: (s > 1).sum())
))

print("В разрезе аптек:")
display(shop_counts)

=== По всем данным ===
Всего уникальных товаров: 6497
Товаров с 1 закупочной ценой: 2886
Товаров с >1 закупочной ценой: 3611
Товаров с 1 розничной ценой: 2903
Товаров с >1 розничной ценой: 3594
В разрезе аптек:


,shop,sku_count,zak_fixed,zak_var,roz_fixed,roz_var
0,Аптека 1,1864,1249,615,1277,587
1,Аптека 10,1819,1254,565,1285,534
2,Аптека 11,2213,1565,648,1595,618
3,Аптека 2,2313,1388,925,1440,873
4,Аптека 3,1837,1293,544,1327,510
5,Аптека 4,1244,883,361,897,347
6,Аптека 7,2146,1414,732,1450,696
7,Аптека 8,1464,966,498,996,468


### 🕵🏻 Наблюдение 11
- Из 6496 уникальных товаров только 2886 имеют одну закупочную цену, а 3610 - больше одной.
- По рознице картина почти такая же: 2903 товара с одной ценой и 3593 с несколькими.
- Это означает, что в базе **у большинства SKU цены менялись хотя бы один раз**.
- Во всех аптеках доля товаров с изменяющимися ценами довольно высокая. Это подтверждает, что различия цен есть не только между товарами, но и внутри точек продаж.  

**Цены динамические**

### Метрики ABC анализа. Датафрейм  `product_df`
Принципы и расчет:  
- Группировка по артикулу `dr_cdrugs` (написания названий могут отличатся).
- Сумма продаж для каждой позиции - `sales_amount = dr_croz * dr_kol`.
- Используется среднее значение закупочной и розничной цены для каждого артикула, т.к. установлено, что цены на один и тот же товар могут быть разными.
- Используем простое суммирование по `dr_kol`, чтобы понять объем продаж конкретного SKU.
- Выручка - это сумма продаж. Суммируется `sales_amount` для всех строк, относящихся к одному SKU.
- Абсолютная маржа. Показывает среднюю прибыль в денежном выражении с одной единицы товара. Вычисляется как разница между средней розничной и средней закупочной ценой.
- Относительная маржа (в процентах). Показывает процент прибыли от розничной цены. Это нормированный показатель, который удобно использовать для сравнения прибыльности разных товаров независимо от их абсолютной стоимости.

In [48]:
product_df = sales_final.copy()

# Рассчитываем выручку для каждой строки перед группировкой
product_df['sales_amount'] = product_df['dr_croz'] * product_df['dr_kol']

# Группируем sales_final для подсчета уникальных аптек для каждого товара
shops_per_product = sales_final.groupby('dr_cdrugs')['shop'].nunique().reset_index(name='num_shops')

product_df = (
    product_df.groupby('dr_cdrugs', as_index=False)
    .agg(
        dr_ndrugs=('dr_ndrugs', 'first'),
        mean_czak=('dr_czak', 'mean'), # меняется, берем среднее
        mean_croz=('dr_croz', 'mean'), # меняется, берем среднее
        total_kol=('dr_kol', 'sum'),
        revenue=('sales_amount', 'sum')
    )
)

# Присоединяем информацию о количестве аптек к product_df
product_df = pd.merge(product_df, shops_per_product, on='dr_cdrugs', how='left')

# Рассчитываем абсолютную и относительную маржу
product_df['abs_margin'] = product_df['mean_croz'] - product_df['mean_czak']
product_df['rel_margin'] = (product_df['abs_margin'] / product_df['mean_croz']) * 100.00

print("Датафрейм product_df создан:")
pd.options.display.float_format = '{:,.2f}'.format
display(product_df.head())

Датафрейм product_df создан:


,dr_cdrugs,dr_ndrugs,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin
0,3,БАНЕОЦИН 10Г. №1 ПОР. Д/НАРУЖ.ПРИМ. ФЛ./ДОЗ.,412.67,549.42,12.00,"6,593.00",6,136.75,24.89
1,25,ОТИПАКС 10МГ/Г+40МГ/Г. 15МЛ/16Г. КАПЛИ УШНЫЕ ФЛ./КАП.,372.11,496.60,20.00,"9,932.00",7,124.49,25.07
2,30,СТРЕПТОЦИД 5% 30Г. ЛИНИМЕНТ ТУБА /НИЖФАРМ/,78.75,114.00,3.00,342.00,3,35.25,30.92
3,31,"ТАНТУМ ВЕРДЕ 0,255МКГ/ДОЗА 30МЛ. 176ДОЗ СПРЕЙ Д/МЕСТ.ПРИМ. ДОЗИР. ФЛ.",344.12,442.41,42.00,"18,581.22",8,98.29,22.22
4,35,ТОНЗИЛОТРЕН №60 ТАБ. Д/РАСС.,727.61,929.12,8.00,"7,433.00",5,201.52,21.69


In [49]:
product_df.describe()

,dr_cdrugs,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin
count,"6,497.00","6,497.00","6,497.00","6,497.00","6,497.00","6,497.00","6,497.00","6,497.00"
mean,"223,455.43",383.71,483.09,7.44,"1,944.60",2.29,99.38,25.19
std,"189,159.62",642.92,741.06,26.08,"3,869.34",1.82,125.96,9.78
min,3.00,0.96,1.00,0.02,1.00,1.00,0.04,2.44
25%,"41,481.00",78.29,108.57,1.00,258.00,1.00,25.60,17.95
50%,"173,365.00",195.92,267.00,2.00,690.00,1.00,61.70,25.98
75%,"362,139.00",442.12,570.67,6.00,"1,970.00",3.00,125.77,34.33
max,"1,504,483.00","17,734.10","18,281.30","1,091.00","58,872.00",8.00,"2,396.89",51.00


### 🕵🏻 Наблюдение 12
- У нас присутсвует 6497 уникальной позиции, проданной за наш период чуть больше месяца.
- Средние закупочные и розничные цены демонстрируют высокую вариативность (стандартное откл.почти в 2 раза выше среднего, разница среднего/медианы, минимума/максимума тоже большие). Подтверждает наличие как дешевых, так и дорогих позиций.
- По количеству позиций видно,  что лишь небольшая часть товаров продается в очень больших объемах, в то время как большинство SKU имеют низкие объемы продаж. Это выглядит логично для аптечной сети.
- Распределение выручки сильно смещено, показывая, что некоторые товары приносят значительно больше, чем остальные (максимум 58 872.00), что крайне важно для ABC-анализа.
- Статистика маржинальности указывает на существенные различия в прибыльности товаров.
- Не все товары продаются во всех аптеках. В среднем позиция продается в 2-х точках, половина товаров продается только в одной точке, но есть позиции, представленные во всех аптеках сети.  

**❗ Эти данные подтверждают, что ассортимент разнообразен, объемы продаж и маржинальность значительно отличаются между товарами, а их представленность в аптеках также варьируется, что необходимо учитывать при дальнейшем анализе**.

### Распределение товаров по количеству аптек

In [50]:
# Определяем общее количество уникальных аптек
total_unique_shops = sales_final['shop'].nunique()
print(f"Общее количество уникальных аптек: {total_unique_shops}")

# 1. Товары, продающиеся только в одной аптеке
single_shop_products = product_df[product_df['num_shops'] == 1].copy()

# Для этих товаров нужно определить, в какой именно аптеке они продаются.
# Сгруппируем sales_final по dr_cdrugs и возьмем название аптеки
single_shop_info = sales_final[sales_final['dr_cdrugs'].isin(single_shop_products['dr_cdrugs'])].groupby('dr_cdrugs')['shop'].first().reset_index()
single_shop_info.rename(columns={'shop': 'selling_shop'}, inplace=True)

single_shop_products = pd.merge(single_shop_products, single_shop_info, on='dr_cdrugs', how='left')

print(f"\nКоличество товаров, продающихся только в одной аптеке: {len(single_shop_products)}")
print("Примеры товаров, продающихся только в одной аптеке:")
display(single_shop_products.head())

# Распределение этих товаров по аптекам
print("\nРаспределение товаров, продающихся только в одной аптеке, по конкретным аптекам:")
display(single_shop_products['selling_shop'].value_counts())

# 2. Товары, продающиеся во всех аптеках
all_shops_products = product_df[product_df['num_shops'] == total_unique_shops]
print(f"\nКоличество товаров, продающихся во всех {total_unique_shops} аптеках: {len(all_shops_products)}")
print("Примеры товаров, продающихся во всех аптеках:")
display(all_shops_products.head())

Общее количество уникальных аптек: 8

Количество товаров, продающихся только в одной аптеке: 3273
Примеры товаров, продающихся только в одной аптеке:


,dr_cdrugs,dr_ndrugs,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin,selling_shop
0,156,АЦИДИН-ПЕПСИН 250МГ. №50 ТАБ. /БЕЛМЕДПРЕПАРАТЫ/,121.00,166.00,1.00,166.00,1,45.00,27.11,Аптека 11
1,174,БЕКАРБОН №6 ТАБ. /ТАТХИМФАРМ/,28.53,44.00,1.00,44.00,1,15.47,35.16,Аптека 7
2,186,БЕТАДИН 10% 20Г. №1 МАЗЬ Д/НАРУЖ.ПРИМ. ТУБА,241.89,327.00,2.00,654.00,1,85.11,26.03,Аптека 1
3,253,БРОНХОЛИТИН 125МЛ. СИРОП ФЛ.,237.71,249.00,1.00,249.00,1,11.29,4.53,Аптека 4
4,293,ВИПРОСАЛ В 50Г. МАЗЬ Д/НАРУЖ. ПРИМ. ТУБА,326.40,425.00,1.00,425.00,1,98.60,23.20,Аптека 11



Распределение товаров, продающихся только в одной аптеке, по конкретным аптекам:


,count
selling_shop,
Аптека 7,565
Аптека 11,520
Аптека 2,493
Аптека 1,412
Аптека 10,396
Аптека 3,390
Аптека 8,271
Аптека 4,226



Количество товаров, продающихся во всех 8 аптеках: 153
Примеры товаров, продающихся во всех аптеках:


,dr_cdrugs,dr_ndrugs,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin
3,31,"ТАНТУМ ВЕРДЕ 0,255МКГ/ДОЗА 30МЛ. 176ДОЗ СПРЕЙ Д/МЕСТ.ПРИМ. ДОЗИР. ФЛ.",344.12,442.41,42.00,"18,581.22",8,98.29,22.22
10,83,АЛМАГЕЛЬ А 170МЛ. №1 СУСП. Д/ПРИЕМА ВНУТРЬ ФЛ. /БАЛКАН ФАРМА-ТРОЯН/,298.42,364.42,24.00,"8,746.00",8,66.00,18.11
18,245,БРОМГЕКСИН 8 БЕРЛИН-ХЕМИ 8МГ. №25 ТАБ. П/О /БЕРЛИН ХЕМИ/,139.55,192.29,41.00,"7,884.00",8,52.74,27.43
22,269,ВАЛОКОРДИН 20МЛ. КАПЛИ Д/ПРИЕМА ВНУТРЬ ФЛ./КАП.,160.78,197.71,58.00,"11,467.00",8,36.93,18.68
25,279,ВЕРОШПИРОН 25МГ. №20 ТАБ. /ГЕДЕОН РИХТЕР/,78.65,95.77,102.00,"9,768.32",8,17.11,17.87


- 3273 товара продаются только в одной аптеке, 153 - во всех 8 аптеках.
- В аптеках есть от 226 до 565 товаров, которые продаются только там и не продавались в других точках (за рассматриваемый период).

### <a id="resume"></a>📌 Краткое резюме перед товарным анализом
На текущем этапе у нас есть данные по продажам за период чуть больше месяца, поэтому все выводы нужно трактовать как наблюдения за анализируемый период, а **не как устойчивую закономерность**. При этом уже видно, что **сеть неоднородна**: аптеки отличаются по масштабу, по структуре продаж, по выручке и по доле покупок по бонусным картам. Это значит, что единый подход (среднее по сети) здесь будет слишком грубым.  

Анализ данных о транзакциях с картами лояльности `bonuscheques` (11 месяцев) показал, что есть точки-лидеры и точки с более слабой лояльностью.  
- Аптека 10 выделяется самой высокой долей идентифицированных карт и высокой частотой покупок на клиента.
- Аптека 2 - крупнейшая по базе и общей выручке.
- Аптека 11 входит в число лидеров по объему, при этом отличается по структуре поведения. Она только появилась в данных и пока ее нужно интерпретировать осторожно.
- Аптеки 1, 4 и 7, наоборот, показывают более низкую долю идентифицированных карт.
- Аптека 8 выглядит как средняя и достаточно стабильная точка.
- Аптека 6, вероятно, выбивается из общего ряда из-за неполного периода и закрылась.  
- Аптека 3 не встечается в транзакциях, но есть в продажах, что тоже требует осторожной интерпретации.

Дополнительно мы увидели, что пиковые часы у аптек совпадают (кроме Аптеки 10, там вечерний пик начинается чуть раньше и сохраняется дольше, чем у других аптек). По дням недели продажи распределяются неравномерно: у большинства аптек пик приходится на середину недели, а выходные слабее. При этом есть и локальные особенности: например, у Аптеки 10 в четверг растет количество чеков, но снижается выручка, а Аптеки 2 и 11 входят в число лидеров, но меняются местами в зависимости от того, смотрим мы на выручку или на число чеков. Это уже показывает, что одних агрегатов по сети недостаточно, важно учитывать и поведенческий профиль точки, и структуру ее продаж.  

На уровне товара мы уже видим, что ассортимент тоже неоднороден. Часть SKU продается только в одной аптеке, часть - в нескольких, и только небольшая доля представлена во всех 8 аптеках. При этом у многих товаров цены не фиксированы: одна и та же позиция может иметь несколько закупочных и розничных цен. Значит, товарный анализ нельзя строить только по средней цене или только по суммарной выручке - нужно отдельно смотреть распространенность товара по сети, ценовую стабильность и маржинальность.  

### 🕵🏻 Общее наблюдение
1. Сеть неоднородна: у аптек различаются масштабы и структура продаж.
2. Ассортимент тоже неоднороден: есть локальные товары, сетевые товары и товары с меняющимися ценами.
3. Для рекомендаций по рассылкам важны не только продажи, но и широта распространения товара, маржа и связь с конкретной аптекой.

## Этап 3.1. Подбор товаров под маркетинговые рассылки
Нам важно понять, какие позиции можно рекомендовать массово, какие - только в отдельных аптеках, какие - лучше использовать для допродаж, а какие вообще не стоит продвигать без дополнительной проверки.  

Определим какие группы товаров подходят для:
- массовых рассылок
- рассылок по конкретным аптекам
- предложений клиентам с высокой частотой покупок
- допродаж клиентам с низкой частотой
- промо товаров с высокой маржой

### Ассортимент сети


In [51]:
# Выручка на каждой строке
sales_final = sales_final.copy()
sales_final['row_revenue'] = sales_final['dr_kol'] * sales_final['dr_croz']

# 1. Сколько аптек продают каждый товар
sku_coverage = (
    sales_final.groupby('dr_cdrugs', as_index=False)
    .agg(
        dr_ndrugs=('dr_ndrugs', 'first'),
        num_shops=('shop', 'nunique'),
        revenue=('row_revenue', 'sum')
    ))

# 2. Сколько SKU в каждой группе по широте присутствия
sku_coverage['coverage_group'] = pd.cut(
    sku_coverage['num_shops'],
    bins=[0, 1, 7, 8],
    labels=['только 1 аптека', '2-7 аптек', 'все 8 аптек'],
    include_lowest=True
)

coverage_summary = (
    sku_coverage.groupby('coverage_group', observed=False, as_index=False)
    .agg(
        sku_count=('dr_cdrugs', 'nunique'),
        revenue=('revenue', 'sum')
    ))

coverage_summary['share_sku_%'] = (
    coverage_summary['sku_count'] / coverage_summary['sku_count'].sum() * 100.00
)

coverage_summary['share_revenue_%'] = (
    coverage_summary['revenue'] / coverage_summary['revenue'].sum() * 100.00
).round(1)

display(coverage_summary)

# 3. Визуализация
fig = px.bar(
    coverage_summary,
    x='coverage_group',
    y='sku_count',
    text='sku_count',
    title='Распределение SKU по широте присутствия в сети'
)

fig.update_xaxes(title_text='Группа товаров')
fig.update_yaxes(title_text='Количество SKU')
fig.show()

,coverage_group,sku_count,revenue,share_sku_%,share_revenue_%
0,только 1 аптека,3273,"2,291,243.25",50.38,18.10
1,2-7 аптек,3071,"8,449,483.85",47.27,66.90
2,все 8 аптек,153,"1,893,326.00",2.35,15.00


### 🕵🏻 Наблюдение 13
Анализ структуры ассортимента за наблюдаемый период (40 дней) показал, что товарное предложение сети неоднородно по широте присутствия в аптеках. Существенная доля SKU реализуется только в одной точке, сопоставимая доля встречается в нескольких аптеках, тогда как позиции, представленные во всей сети, составляют лишь небольшую часть ассортимента. Полученные результаты следует интерпретировать как характеристику ограниченного периода наблюдений, а не как окончательную оценку устойчивости ассортимента.

### Товарная значимость позиций

In [52]:
product_summary = (
    sales_final.groupby('dr_cdrugs', as_index=False)
    .agg(
        dr_ndrugs=('dr_ndrugs', 'first'),
        revenue=('row_revenue', 'sum'),
        total_kol=('dr_kol', 'sum'),
        num_checks=('check_id', 'nunique'),
        num_shops=('shop', 'nunique')
    )
)

product_summary = product_summary.sort_values('revenue', ascending=False)

display(product_summary)

,dr_cdrugs,dr_ndrugs,revenue,total_kol,num_checks,num_shops
2149,76851,КОНВАЛИС 300МГ. №50 КАПС.,"58,872.00",80.00,66,1
2858,142036,ПЕНТАЛГИН №24 ТАБ. П/П/О /ОТИСИФАРМ/,"56,782.00",261.00,249,8
3412,192460,"ЭЛИКВИС 2,5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-МАЙЕРС/","55,986.44",22.00,21,5
808,13574,НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ВНУТРЬ ПАК. /ГУИДОТТИ/МЕНАРИНИ/,"54,460.63",41.57,179,8
4376,338370,ДЕТРАЛЕКС 1000МГ. №60 ТАБ. П/П/О,"53,984.00",17.00,17,8
...,...,...,...,...,...,...
3005,154972,АСКОРБИНОВАЯ К-ТА С ГЛЮКОЗОЙ ГЛЕНВИТОЛ ГРАНАТ+ЧЕРНИКА 25МГ. №10 ТАБ.ЖЕВ. КРУТКА,15.00,1.00,1,1
2378,98912,МОЧЕПРИЕМНИК ДЕТСКИЙ 200МЛ.,13.00,1.00,1,1
4230,323711,АСКОРБИНОВАЯ К-ТА С ГЛЮКОЗОЙ ГЛЕНВИТОЛ 25МГ. №10 ТАБ. КРУТКА (БАД),10.00,1.00,1,1
5238,404562,БЕНОВИ ПЕРЧАТКИ СМОТР. ВИНИЛ Н/СТЕР. ПРОЗРАЧ. Р.M №100 (50 ПАР) [BENOVY],6.74,0.02,1,1


Посмотрим топ-10 по метрикам

In [53]:
# Выручка
top20_revenue = (
    product_summary
    .sort_values('revenue', ascending=False)
    .head(20)
)

print("Выручка")
display(top20_revenue)

# Количество
top20_qty = (
    product_summary
    .sort_values('total_kol', ascending=False)
    .head(20)
)

print("Количество")
display(top20_qty)

# Число чеков
top20_checks = (
    product_summary
    .sort_values('num_checks', ascending=False)
    .head(20)
)

print("Число чеков")
display(top20_checks)

Выручка


,dr_cdrugs,dr_ndrugs,revenue,total_kol,num_checks,num_shops
2149,76851,КОНВАЛИС 300МГ. №50 КАПС.,"58,872.00",80.00,66,1
2858,142036,ПЕНТАЛГИН №24 ТАБ. П/П/О /ОТИСИФАРМ/,"56,782.00",261.00,249,8
3412,192460,"ЭЛИКВИС 2,5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-МАЙЕРС/","55,986.44",22.00,21,5
808,13574,НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ВНУТРЬ ПАК. /ГУИДОТТИ/МЕНАРИНИ/,"54,460.63",41.57,179,8
4376,338370,ДЕТРАЛЕКС 1000МГ. №60 ТАБ. П/П/О,"53,984.00",17.00,17,8
6018,517544,ГАБАПЕНТИН 300МГ. №100 КАПС. /КАНОНФАРМА/,"52,823.00",60.00,58,1
3393,190635,ЭЛИКВИС 5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-МАЙЕРС/,"51,668.00",20.00,19,7
3099,162190,КСАРЕЛТО 20МГ. №28 ТАБ. П/П/О /БАЙЕР/,"49,554.50",15.50,16,6
1035,20433,ЭНТЕРОСГЕЛЬ 225Г. ПАСТА Д/ПРИЕМА ВНУТРЬ ТУБА,"41,664.00",87.00,83,8
215,2302,ОМЕЗ 20МГ. №30 КАПС. /Д-Р РЕДДИ/,"39,989.77",225.00,205,8


Количество


,dr_cdrugs,dr_ndrugs,revenue,total_kol,num_checks,num_shops
6488,1504015,ПАКЕТ,"2,182.00","1,091.00",1084,8
1490,33001,"ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 2,5Х7,2 №1 /ВЕРОФАРМ/","3,354.16",862.00,73,5
4768,348186,"НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ ФЛ./КАП. /СЛАВЯНСКАЯ АПТЕКА/","5,766.00",582.00,83,6
5159,393397,КОРВАЛОЛ 25МЛ. КАПЛИ Д/ПРИЕМА ВНУТРЬ ФЛ. И/У /ФАРМСТАНДАРТ ЛЕКСРЕДСТВА/,"9,242.35",351.00,195,8
2072,72392,"СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТАДА/","38,286.68",319.00,262,8
5878,496504,"НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ ФЛ./КАП. /ЛЕККО/","5,393.00",296.00,99,5
5306,418134,ВАЛИДОЛ 60МГ. №10 ТАБ. ПОДЪЯЗЫЧ. /ФАРМСТАНДАРТ/,"10,129.00",276.00,131,7
488,6402,ФЕРРОГЕМАТОГЕН 50Г. ПАСТИЛКА ЖЕВ. (ПЛИТКА),"12,518.00",273.00,141,8
4150,314970,БОРНАЯ К-ТА 10Г. ДЕЗ.СР-ВО ПОР. (НДС 20%),"5,253.00",271.00,73,7
2858,142036,ПЕНТАЛГИН №24 ТАБ. П/П/О /ОТИСИФАРМ/,"56,782.00",261.00,249,8


Число чеков


,dr_cdrugs,dr_ndrugs,revenue,total_kol,num_checks,num_shops
6488,1504015,ПАКЕТ,"2,182.00","1,091.00",1084,8
2072,72392,"СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТАДА/","38,286.68",319.00,262,8
2858,142036,ПЕНТАЛГИН №24 ТАБ. П/П/О /ОТИСИФАРМ/,"56,782.00",261.00,249,8
215,2302,ОМЕЗ 20МГ. №30 КАПС. /Д-Р РЕДДИ/,"39,989.77",225.00,205,8
5159,393397,КОРВАЛОЛ 25МЛ. КАПЛИ Д/ПРИЕМА ВНУТРЬ ФЛ. И/У /ФАРМСТАНДАРТ ЛЕКСРЕДСТВА/,"9,242.35",351.00,195,8
5011,377588,"ТИЗИН КЛАССИК 0,1% 10МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ДЖОНСОН/","25,401.00",236.00,192,8
4913,365627,"РИНОНОРМ-ТЕВА 0,1% 20МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ТЕВА/","24,235.00",245.00,181,8
808,13574,НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ВНУТРЬ ПАК. /ГУИДОТТИ/МЕНАРИНИ/,"54,460.63",41.57,179,8
973,18836,ТАУФОН 4% 10МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП. /ОТИСИФАРМ/ФАРМСТАНДАРТ/,"24,020.64",191.00,172,8
5651,463100,КЕТОРОЛ ЭКСПРЕСС 10МГ. №20 ТАБ. ДИСПЕРГ. /Д-Р РЕДДИС/,"13,278.00",194.00,159,8


**Логика отбора**:  
1. Составим множество из полученных топов, исключая пакет.
2. Выделим 5 групп  

| Группа   | Признаки  |
|----------|-----------|
| Массовые рассылки | в топ-20 по выручке и/или по числу чеков + продаются во многих аптеках сети (5+) |
| Лояльная аудитория | высокая средняя розничная цена (> 66-го перц. 1389,38) |
| Предложения клиентам с высокой частотой покупок | в топ-20 по числу чеков + меньше медианы средней розничной цены (< med 125.76) + отн.маржа (> 33-го перц. 16.66), рекомендация выгодна для аптеки, но не отпугивает ценой пок-ля |
| Допродажи клиентам с низкой частотой  | высокомаржинальные (> avg 22.84) + высокая средняя розничная цена (> avg 647.78), стремимся получить максимум с каждой редкой покупки |
| Промо товаров с высокой маржой  | относительная маржа выше 66-го перцентиля (24.13) |



In [54]:
# product_df.head()

In [55]:
# исключаем пакет из анализа
product_summary_no_pack = product_summary[product_summary['dr_ndrugs'] != 'ПАКЕТ'].copy()

# топы без пакета
top20_revenue = (
    product_summary_no_pack
    .sort_values('revenue', ascending=False)
    .head(20)
)

top20_qty = (
    product_summary_no_pack
    .sort_values('total_kol', ascending=False)
    .head(20)
)

top20_checks = (
    product_summary_no_pack
    .sort_values('num_checks', ascending=False)
    .head(20)
)

# длинный формат: каждый товар может попадать в несколько топов
top_products_long = pd.concat([
    top20_revenue[['dr_cdrugs', 'dr_ndrugs']].assign(top_metric='revenue'),
    top20_qty[['dr_cdrugs', 'dr_ndrugs']].assign(top_metric='total_kol'),
    top20_checks[['dr_cdrugs', 'dr_ndrugs']].assign(top_metric='num_checks')
], ignore_index=True)

# агрегируем список метрик для каждого товара
top_products = (
    top_products_long
    .groupby(['dr_cdrugs', 'dr_ndrugs'], as_index=False)
    .agg(
        top_metrics=('top_metric', lambda s: ', '.join(sorted(set(s))))
    ))

# подтягиваем товарные метрики
top_products = top_products.merge(
    product_df[['dr_cdrugs', 'mean_czak', 'mean_croz', 'total_kol', 'revenue', 'num_shops', 'abs_margin', 'rel_margin']],
    on='dr_cdrugs',
    how='left'
)

# флаги под будущую разметку
top_products['mass_mailings'] = 0
top_products['shop_mailings'] = 0
top_products['high_freq_clients'] = 0
top_products['low_freq_clients'] = 0
top_products['high_margin_promo'] = 0
top_products['segment'] = ''

print(f"Всего отобрано {top_products['dr_cdrugs'].nunique()} тов.")
display(top_products)

Всего отобрано 43 тов.


,dr_cdrugs,dr_ndrugs,top_metrics,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin,mass_mailings,shop_mailings,high_freq_clients,low_freq_clients,high_margin_promo,segment
0,2302,ОМЕЗ 20МГ. №30 КАПС. /Д-Р РЕДДИ/,"num_checks, revenue, total_kol",147.93,177.73,225.00,"39,989.77",8,29.80,16.77,0,0,0,0,0,
1,5571,НАЗОНЕКС 50МКГ/ДОЗА 18Г. 120ДОЗ №1 НАЗАЛ.СПРЕЙ ФЛ.,revenue,905.20,"1,171.38",32.00,"37,484.00",7,266.17,22.72,0,0,0,0,0,
2,6362,ГЕПТРАЛ 400МГ. №20 ТАБ.КШ/РАСТВ. П/О /ЭББОТ/,revenue,"1,679.18","1,852.62",16.00,"29,642.00",5,173.45,9.36,0,0,0,0,0,
3,6402,ФЕРРОГЕМАТОГЕН 50Г. ПАСТИЛКА ЖЕВ. (ПЛИТКА),"num_checks, total_kol",28.09,45.62,273.00,"12,518.00",8,17.53,38.43,0,0,0,0,0,
4,13574,НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ВНУТРЬ ПАК. /ГУИДОТТИ/МЕНАРИНИ/,"num_checks, revenue","1,026.98","1,326.66",41.57,"54,460.63",8,299.67,22.59,0,0,0,0,0,
5,16026,КАНЕФРОН Н №60 ТАБ. П/О,revenue,577.87,661.71,52.00,"34,409.00",8,83.84,12.67,0,0,0,0,0,
6,16911,"КАРДИОМАГНИЛ 75МГ.+15,2МГ. №100 ТАБ.","num_checks, revenue",237.30,276.21,131.00,"36,183.32",8,38.91,14.09,0,0,0,0,0,
7,17182,ПОЛИДЕКСА С ФЕНИЛЭФРИНОМ 15МЛ. НАЗАЛ.СПРЕЙ ФЛ.,revenue,487.72,647.33,53.00,"34,322.00",8,159.61,24.66,0,0,0,0,0,
8,18836,ТАУФОН 4% 10МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП. /ОТИСИФАРМ/ФАРМСТАНДАРТ/,"num_checks, total_kol",112.28,125.76,191.00,"24,020.64",8,13.48,10.72,0,0,0,0,0,
9,20433,ЭНТЕРОСГЕЛЬ 225Г. ПАСТА Д/ПРИЕМА ВНУТРЬ ТУБА,revenue,392.85,478.90,87.00,"41,664.00",8,86.05,17.97,0,0,0,0,0,


Распеделять буду вручную. Специально взяты топы, а не все, чтобы это было возможно (правила четко зафиксированы выше)

In [56]:
# в Excel
# top_products.to_excel('top_products.xlsx', index=False)
# print("DataFrame 'top_products' saved to 'top_products.xlsx'")

Проверим наши товары на то, часто ли и сильно ли меняются цены на них (в тако случае их лучше исключить из массовой рассылки), а также добавим информацию о конкретных аптеках, где они продавались

In [57]:
sf = sales_final.copy()

sku_list = top_products['dr_cdrugs'].unique()
sub = sf[sf['dr_cdrugs'].isin(sku_list)].copy()

price_stats = (
    sub.groupby('dr_cdrugs', as_index=False)
    .agg(
        dr_ndrugs=('dr_ndrugs', 'first'),
        zak_unique=('dr_czak', 'nunique'),
        roz_unique=('dr_croz', 'nunique'),
        zak_min=('dr_czak', 'min'),
        zak_max=('dr_czak', 'max'),
        roz_min=('dr_croz', 'min'),
        roz_max=('dr_croz', 'max'),
        shops_list=('shop', lambda s: ', '.join(sorted(set(s)))),
        shop_count=('shop', 'nunique')
    )
)

rows_stats = (
    sub.groupby('dr_cdrugs', as_index=False)
    .agg(rows=('dr_cdrugs', 'size'))
)

price_stats = price_stats.merge(rows_stats, on='dr_cdrugs', how='left')

price_stats['stable_zak'] = price_stats['zak_unique'].eq(1)
price_stats['stable_roz'] = price_stats['roz_unique'].eq(1)
price_stats['price_change_flag'] = (~price_stats['stable_zak']) | (~price_stats['stable_roz'])

if 'segment' in top_products.columns:
    price_stats = price_stats.merge(
        top_products[['dr_cdrugs', 'segment']],
        on='dr_cdrugs',
        how='left'
    )

cols = [
    'dr_cdrugs', 'dr_ndrugs', 'segment',
    'zak_unique', 'roz_unique',
    'zak_min', 'zak_max', 'roz_min', 'roz_max',
    'shop_count', 'shops_list', 'rows'
]
cols = [c for c in cols if c in price_stats.columns]

price_stats = price_stats[cols].sort_values(
    by=['zak_unique', 'roz_unique', 'shop_count'],
    ascending=[False, False, False]
)

display(price_stats)

,dr_cdrugs,dr_ndrugs,segment,zak_unique,roz_unique,zak_min,zak_max,roz_min,roz_max,shop_count,shops_list,rows
16,72392,"СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТАДА/",,28,16,95.88,107.76,112.00,127.00,8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",312
4,13574,НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ВНУТРЬ ПАК. /ГУИДОТТИ/МЕНАРИНИ/,,27,41,959.30,"1,283.84","1,138.00","1,616.00",8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",187
34,377588,"ТИЗИН КЛАССИК 0,1% 10МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ДЖОНСОН/",,25,13,87.60,102.85,103.00,121.00,8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",236
32,354089,УГОЛЬ АКТИВИРОВАННЫЙ 250МГ. №50 ТАБ. /ФАРМСТАНДАРТ ЛЕКСРЕДСТВА/,,24,15,31.52,46.72,48.00,74.00,7,"Аптека 1, Аптека 10, Аптека 11, Аптека 3, Аптека 4, Аптека 7, Аптека 8",179
3,6402,ФЕРРОГЕМАТОГЕН 50Г. ПАСТИЛКА ЖЕВ. (ПЛИТКА),,24,12,25.10,54.42,41.00,57.00,8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",149
27,318348,"СФМ ШПРИЦ 5МЛ. 3-Х КОМП. 0,7Х40ММ 22G №10 [SFM]",,23,19,103.49,213.31,161.00,329.00,7,"Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",134
30,345578,ПАРАЦЕТАМОЛ 500МГ. №20 ТАБ. /ФАРМСТАНДАРТ/,,23,6,14.52,23.91,15.21,29.00,8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",179
6,16911,"КАРДИОМАГНИЛ 75МГ.+15,2МГ. №100 ТАБ.",,22,21,214.03,333.30,228.00,349.00,8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",131
10,22533,"КОНКОР КОР 2,5МГ. №30 ТАБ. П/П/О /МЕРК/",,22,19,130.04,146.10,150.00,173.00,8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",164
20,142036,ПЕНТАЛГИН №24 ТАБ. П/П/О /ОТИСИФАРМ/,,22,19,176.42,213.94,207.00,246.00,8,"Аптека 1, Аптека 10, Аптека 11, Аптека 2, Аптека 3, Аптека 4, Аптека 7, Аптека 8",261


In [58]:
# в Excel
# price_stats.to_excel('price_stats.xlsx', index=False)
# print("DataFrame 'price_stats' saved to 'price_stats.xlsx'")

In [59]:
product_df

,dr_cdrugs,dr_ndrugs,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin
0,3,БАНЕОЦИН 10Г. №1 ПОР. Д/НАРУЖ.ПРИМ. ФЛ./ДОЗ.,412.67,549.42,12.00,"6,593.00",6,136.75,24.89
1,25,ОТИПАКС 10МГ/Г+40МГ/Г. 15МЛ/16Г. КАПЛИ УШНЫЕ ФЛ./КАП.,372.11,496.60,20.00,"9,932.00",7,124.49,25.07
2,30,СТРЕПТОЦИД 5% 30Г. ЛИНИМЕНТ ТУБА /НИЖФАРМ/,78.75,114.00,3.00,342.00,3,35.25,30.92
3,31,"ТАНТУМ ВЕРДЕ 0,255МКГ/ДОЗА 30МЛ. 176ДОЗ СПРЕЙ Д/МЕСТ.ПРИМ. ДОЗИР. ФЛ.",344.12,442.41,42.00,"18,581.22",8,98.29,22.22
4,35,ТОНЗИЛОТРЕН №60 ТАБ. Д/РАСС.,727.61,929.12,8.00,"7,433.00",5,201.52,21.69
...,...,...,...,...,...,...,...,...,...
6492,1504476,АЕВИТ КАПС. №30*,59.89,86.84,9.00,781.56,1,26.95,31.03
6493,1504477,"НАРИНЭ КАПСУЛЫ, 20 ШТ. НАРЭКС",304.62,319.00,1.00,319.00,1,14.38,4.51
6494,1504481,ТЕСТ НА БЕРЕМЕННОСТЬ ВЫСОКОЧУВСТВИТЕЛЬНЫЙ 1 ШТ. КЛЕВЕР,22.99,24.11,1.00,24.11,1,1.12,4.65
6495,1504482,"ЧУЛОК КОМПРЕССИОННЫЙ (ДО КОЛЕНА) РАЗ.4, ЛПП ФАРМ",267.36,280.00,1.00,280.00,1,12.64,4.51


### 📌 Рекомендации 1 (без одноразовых покупателей)
На предыдущем этапе был сформирован пул кандидатов для рекомендательных предложений на основе данных о продажах аптечной сети за 40-дневный период. Далее этот список был детально проанализирован вручную, в результате чего выработаны целевые рекомендации для разных сегментов клиентской базы.  
[Ссылка на файл excel](https://docs.google.com/spreadsheets/d/1Xrs7_0OviyqU_2my-xqPDpOq_AIpY5qc/edit?usp=sharing&ouid=117814435472598854510&rtpof=true&sd=true)  

При формировании исходного списка кандидатов применялся следующий подход: в качестве потенциальных товаров для рекомендаций рассматривались позиции, вошедшие в топ-20 по ключевым метрикам - выручке, количеству проданных единиц и числу чеков. Эти три выборки объединялись во множество (без дубликатов), после чего из него исключались товарные позиции категории «пакет» (несмотря на высокую частоту продаж, они не несут смысловой ценности для рекомендательной логики).  

Дальнейшее распределение товаров по сегментам аудитории проводилось вручную, при этом для каждого сегмента были заданы четкие **критерии**:  
- Массовая рассылка. В эту группу включались товары, одновременно входящие в топ-20 по выручке и/или по числу чеков, а также представленные в ассортименте не менее чем в 5 аптеках сети. Это обеспечивает баланс между востребованностью и доступностью товара для широкой аудитории.
- Лояльная аудитория. Поскольку на текущих данных невозможно точно установить, какой именно клиент что покупал, для этого сегмента использовались косвенные критерии: средняя розничная цена выше 66-го перцентиля в сочетании с попаданием в топ по выручке. Такой подход позволяет предлагать лояльным клиентам товары с более высокой ценовой позицией, сохраняя при этом их коммерческую значимость.
- Часто покупающие. Здесь приоритет отдавался товарам, одновременно входящим в топ-20 по числу чеков, имеющим цену ниже медианы (менее 125.76) и относительную маржинальность выше 33-го перцентиля. Таким образом, рекомендации остаются привлекательными по цене для клиента и одновременно выгодными для аптеки.
- Редко покупающие. Для этого сегмента выбирались товары с высокой относительной маржинальностью (выше среднего значения 22.84) и повышенной средней розничной ценой (выше 647.78). Цель - предложить более дорогие и прибыльные позиции, которые могут быть интересны менее активной аудитории.
- Товары с повышенной относительной маржинальностью. В отдельную группу выделены позиции с относительной маржой выше 66-го перцентиля, при условии, что они представлены в 5 и более аптеках сети. Эта группа служит дополнительным источником высокомаржинальных предложений, которые можно использовать в точечных коммуникациях.  

**Товары для массовой рассылки**  
- приоритет:	РИНОСТОП 0,1% 15МЛ. НАЗАЛ.СПРЕЙ (196444), РИНОНОРМ-ТЕВА 0,1% 20МЛ. НАЗАЛ.СПРЕЙ (365627), КСАРЕЛТО 20МГ. №28 ТАБ. (162190)
- можно:	ОМЕЗ 20МГ. №30 КАПС. (2302), ГЕПТРАЛ 400МГ. №20 ТАБ. (6362), КСАРЕЛТО 10МГ. №30 ТАБ. (168131), ЭЛИКВИС 5МГ. №60 ТАБ. (190635), ЭЛИКВИС 2,5МГ. №60 ТАБ. (192460)
- с осторожностью:	ЦИТРАМОН П №20 ТАБ. (393314), ВАЛИДОЛ 60МГ. №10 ТАБ. ПОДЪЯЗЫЧ. (418134), НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ (496504)  

**Товары для лояльной аудитории**  
ГЕПТРАЛ 400МГ. №20 ТАБ. (6362), НЕБИЛЕТ 5МГ. №28 ТАБ. (60306), КСАРЕЛТО 20МГ. №28 ТАБ. (162190), КСАРЕЛТО 10МГ. №30 ТАБ. (168131), ЭЛИКВИС 5МГ. №60 ТАБ. (190635), ЭЛИКВИС 2,5МГ. №60 ТАБ. (192460), ДЕТРАЛЕКС 1000МГ. №60 ТАБ. (338370)  

**Товары для часто покупающих**  
ФЕРРОГЕМАТОГЕН 50Г. ПАСТИЛКА ЖЕВ. (6402), ЛЮКСПЛАСТ ЛЕЙКОПЛАСТ. БАКТЕР. 19Х72ММ ТКАН. ЭЛ.ТЕЛЕСН. №10 (42974), РОМАШКИ ЦВЕТКИ 1,5Г. №20 ПАК. (46046), РИНОСТОП 0,1% 15МЛ. НАЗАЛ.СПРЕЙ (196444), ПАРАЦЕТАМОЛ 500МГ. №20 ТАБ. (345578), УГОЛЬ АКТИВИРОВАННЫЙ 250МГ. №50 ТАБ.(354089), РИНОНОРМ-ТЕВА 0,1% 20МЛ. НАЗАЛ.СПРЕЙ (365627), ТИЗИН КЛАССИК 0,1% 10МЛ. НАЗАЛ.СПРЕЙ (377588), ЦИТРАМОН П №20 ТАБ. (393314), КОРВАЛОЛ 25МЛ. КАПЛИ Д/ПРИЕМА ВНУТРЬ (393397), КЕТОРОЛ ЭКСПРЕСС 10МГ. №20 ТАБ. (463100)  

**Товары для редко покупающих**  
НАЗОНЕКС 50МКГ/ДОЗА 18Г. 120ДОЗ №1 НАЗАЛ.СПРЕЙ (5571), ГЕПТРАЛ 400МГ. №20 ТАБ. (6362), НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ВНУТРЬ ПАК. (13574), КАНЕФРОН Н №60 ТАБ. (16026), НЕБИЛЕТ 5МГ. №28 ТАБ. (60306), КСАРЕЛТО 20МГ. №28 ТАБ. (162190), КСАРЕЛТО 10МГ. №30 ТАБ. (168131), ЭЛИКВИС 5МГ. №60 ТАБ. (190635), ЭЛИКВИС 2,5МГ. №60 ТАБ. (192460), ДЕТРАЛЕКС 1000МГ. №60 ТАБ. (338370)  

**Товары с повышенной маржинальностью**  
ФЕРРОГЕМАТОГЕН 50Г. ПАСТИЛКА ЖЕВ. (6402), ПОЛИДЕКСА С ФЕНИЛЭФРИНОМ 15МЛ. НАЗАЛ.СПРЕЙ (17182), ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 2,5Х7,2 №1 (33001), ЛЮКСПЛАСТ ЛЕЙКОПЛАСТ. БАКТЕР. 19Х72ММ ТКАН. ЭЛ.ТЕЛЕСН. №10 (42974), РОМАШКИ ЦВЕТКИ 1,5Г. №20 ПАК. (46046), ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 6X10 №1 (81988), БОРНАЯ К-ТА 10Г. ДЕЗ.СР-ВО ПОР. (314970), СФМ ШПРИЦ 5МЛ. 3-Х КОМП. 0,7Х40ММ 22G №10 (318348), НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ (348186), УГОЛЬ АКТИВИРОВАННЫЙ 250МГ. №50 ТАБ. (354089), ЦИТРАМОН П №20 ТАБ. (393314), КЕТОРОЛ ЭКСПРЕСС 10МГ. №20 ТАБ. (463100), НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ (496504)   


Также важно отметить, что в клиентской базе зафиксирована значительная доля одноразовых покупателей. Рекомендации для этой группы будут сформированы отдельно.

### Для совершивших покупку 1 раз
Для клиентов, совершивших одну покупку, персонализация по истории конкретного товара затруднена, так как связь клиента и купленного товара восстановить нельзя. Кроме того, идентифицирована не вся клиентская база (только 55% по `bonuscheques`), поэтому выводы по этой группе следует трактовать как приближенные, но достаточно надежные для построения практических рекомендаций.  

В этой ситуации разумно формировать отдельный пул товаров на основе их рыночных характеристик: популярности, регулярности спроса, широты представленности в аптеках и ценового уровня.  

Для включения в пул рекомендаций товар должен:  
- Быть достаточно популярным по выручке и количеству продаж (`total_kol` + `revenue` не ниже 75-го перц.по выборке).
- Продаваться во всех аптеках, чтобы рекомендация была реалистичной (`'num_shops' >= 8`).
- Иметь умеренный ценовой уровень, чтобы не создавать высокий барьер для повторной покупки (`mean_croz` не выше медианы по выборке).
- Быть выгодным для сети (относительная маржа не ниже 75-го перц.по выборке).

In [60]:
product_df.describe()

,dr_cdrugs,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin
count,"6,497.00","6,497.00","6,497.00","6,497.00","6,497.00","6,497.00","6,497.00","6,497.00"
mean,"223,455.43",383.71,483.09,7.44,"1,944.60",2.29,99.38,25.19
std,"189,159.62",642.92,741.06,26.08,"3,869.34",1.82,125.96,9.78
min,3.00,0.96,1.00,0.02,1.00,1.00,0.04,2.44
25%,"41,481.00",78.29,108.57,1.00,258.00,1.00,25.60,17.95
50%,"173,365.00",195.92,267.00,2.00,690.00,1.00,61.70,25.98
75%,"362,139.00",442.12,570.67,6.00,"1,970.00",3.00,125.77,34.33
max,"1,504,483.00","17,734.10","18,281.30","1,091.00","58,872.00",8.00,"2,396.89",51.00


In [61]:
# Отбор товаров для рекомендаций клиентам с одной покупкой
# Логика: популярность + регулярный спрос + широкое покрытие + умеренная цена
# Маржа используется только для приоритизации внутри списка

product_df_rec = product_df.copy()

q_revenue = product_df_rec['revenue'].quantile(0.75)
q_total_kol = product_df_rec['total_kol'].quantile(0.75)
q_price = product_df_rec['mean_croz'].quantile(0.50)
q_rel_margin = product_df_rec['rel_margin'].quantile(0.75)

candidate_products = (
    product_df_rec[
        (product_df_rec['num_shops'] >= 8) &
        (product_df_rec['revenue'] >= q_revenue) &
        (product_df_rec['total_kol'] >= q_total_kol) &
        (product_df_rec['mean_croz'] <= q_price) &
        (product_df_rec['rel_margin'] >= q_rel_margin)
    ]
    .copy()
)

candidate_products = candidate_products.sort_values(
    by=['revenue', 'total_kol', 'rel_margin'],
    ascending=[False, False, False]
).reset_index(drop=True)

display(candidate_products)

,dr_cdrugs,dr_ndrugs,mean_czak,mean_croz,total_kol,revenue,num_shops,abs_margin,rel_margin
0,61144,"СФМ ШПРИЦ 2МЛ. 3-Х КОМП. 0,63Х32ММ 23G №10 [SFM]",115.00,178.06,78.10,"13,727.55",8,63.06,35.42
1,463100,КЕТОРОЛ ЭКСПРЕСС 10МГ. №20 ТАБ. ДИСПЕРГ. /Д-Р РЕДДИС/,44.31,68.44,194.00,"13,278.00",8,24.13,35.26
2,6402,ФЕРРОГЕМАТОГЕН 50Г. ПАСТИЛКА ЖЕВ. (ПЛИТКА),28.09,45.62,273.00,"12,518.00",8,17.53,38.43
3,42974,ЛЮКСПЛАСТ ЛЕЙКОПЛАСТ. БАКТЕР. 19Х72ММ ТКАН. ЭЛ.ТЕЛЕСН. №10 [LUXPLAST],48.19,78.16,151.00,"11,794.00",8,29.97,38.35
4,341505,ВАЛЕРИАНА ЭКСТРАКТ 20МГ. №50 ТАБ. П/О /БОРИСОВСКИЙ/,44.92,69.75,154.00,"10,742.00",8,24.83,35.60
5,400035,АНАЛЬГИН 500МГ. №20 ТАБ. /ФАРМСТАНДАРТ/,36.82,56.97,115.00,"6,551.00",8,20.14,35.36
6,341996,АСКОРУТИН 50МГ.+50МГ. №50 ТАБ. /ФАРМСТАНДАРТ-УФИМСКИЙ/,42.96,66.81,48.00,"3,207.00",8,23.85,35.70
7,31351,БИНТЛИ-М БИНТ ЛИПКИЙ ПРОНИЦАЕМЫЙ 10СМХ2М.,61.49,96.21,32.00,"3,063.00",8,34.72,36.09
8,38641,АСКОРБИНОВАЯ К-ТА 25МГ. №10 ТАБ. КРУТКА САХ. (БАД),12.00,20.17,142.00,"2,859.00",8,8.17,40.51
9,13441,САЛИПОД ЛЕЙКОПЛАСТ. МОЗОЛЬНЫЙ 6Х10СМ. /ВЕРОФАРМ/,52.26,83.20,32.00,"2,660.00",8,30.94,37.19


Пакет (1504015) формально удовлетворяет всем количественным критериям отбора, но рекомендовать его в персонализированных сообщениях нецелесообразно. Однако, это выгодный и важный для сети товар: стабильная маржа, высокая оборачиваемость, неплохая выручка. Его стоит учитывать при планировании промо-механик, где сопутствующий товар может выступать в роли допродажи на кассе.

### 📌 Рекомендации 2 (для одноразовых покупателей)
Отбор товарных позиций для рекомендаций клиентам с одной покупкой осуществлялся на основе агрегированных характеристик ассортимента. В выборку включались товары с высокими значениями выручки и объема продаж, представленностью не менее чем в восьми аптеках, уровнем цены не выше медианного значения по ассортименту и высокой относительной маржой. Такой подход позволил сформировать компактный список наиболее устойчивых и коммерчески значимых позиций. Технические строки, не относящиеся к фармацевтическому ассортименту (пакет), были исключены из анализа.

**Список товаров для рекомендаций совершившим 1 покупку**  
1. СФМ ШПРИЦ 2МЛ. 3-Х КОМП. 0,63Х32ММ 23G №10 (61144)
2. КЕТОРОЛ ЭКСПРЕСС 10МГ. №20 ТАБ. (463100)
3. ФЕРРОГЕМАТОГЕН 50Г. ПАСТИЛКА ЖЕВ. (6402)
4. ЛЮКСПЛАСТ ЛЕЙКОПЛАСТ. БАКТЕР. 19Х72ММ ТКАН. ЭЛ.ТЕЛЕСН. №10 (42974)
5. ВАЛЕРИАНА ЭКСТРАКТ 20МГ. №50 ТАБ. (341505)
6. АНАЛЬГИН 500МГ. №20 ТАБ. (400035)
7. АСКОРУТИН 50МГ.+50МГ. №50 ТАБ. (341996)
8. БИНТЛИ-М БИНТ ЛИПКИЙ ПРОНИЦАЕМЫЙ 10СМХ2М. (31351)
9. АСКОРБИНОВАЯ К-ТА 25МГ. №10 ТАБ. КРУТКА САХ. (38641)
10. САЛИПОД ЛЕЙКОПЛАСТ. МОЗОЛЬНЫЙ 6Х10СМ. (13441)
11. АСКОРБИНОВАЯ К-ТА 100МГ.+877МГ. ГЛЮКОЗА №40 ТАБ. (38815)
12. ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 6X10 №1 (81988)